# 2-D convolution with a shared-memory tile and halo

A "valid" 2-D cross-correlation of an $H \times W$ image with a $K \times K$ filter produces an $(H-K+1) \times (W-K+1)$ output:

$$y_{r,c} = \sum_{a=0}^{K-1} \sum_{b=0}^{K-1} w_{a,b}\, x_{r+a,\, c+b}$$

One thread per output works, but each input pixel is then fetched from global memory by up to $K^2$ different threads. Here, each block first copies the input region it needs into **shared memory** and computes from there.

## The tiling
- Block `(16, 16)` computes a $16 \times 16$ tile of outputs; grid $(\lceil W_o/16 \rceil, \lceil H_o/16 \rceil)$ with x → columns.
- The tile's outputs read an input patch of $(16 + K - 1) \times (16 + K - 1)$ pixels: the tile plus a **halo** of $K - 1$ extra rows and columns. Allocate `cuda.shared.array((20, 20), float32)` (the size for the largest allowed $K = 5$) and use its top-left corner.
- **Cooperative load**: the patch has more pixels than the block has threads, so number the threads `t = ty * 16 + tx` and let thread `t` load patch entries `t, t + 256, t + 512, ...` (flattened row-major). Pixels outside the image are stored as 0 (they are only ever needed by outputs outside the valid range).
- Also copy the $K \times K$ filter into a `(5, 5)` shared array.
- `cuda.syncthreads()`, then each thread with a valid output computes its sum from shared memory and writes `y`.

## Inputs and output
- `x`: `float32`, shape `(H, W)`; `w`: `float32`, shape `(K, K)` with $1 \le K \le 5$ and $K \le H, W$.
- Return `y`: `float32`, shape `(H-K+1, W-K+1)`. Compared with relative and absolute tolerance $10^{-5}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

TILE, KMAX = 16, 5
PATCH = TILE + KMAX - 1   # 20: shared tile size for the largest filter

def conv2d_valid(x, w):
    """x: float32 (H, W), w: float32 (K, K), K <= 5. Returns float32 (H-K+1, W-K+1)."""
    H, W = x.shape
    K = w.shape[0]
    Ho, Wo = H - K + 1, W - K + 1
    d_x = cuda.to_device(np.ascontiguousarray(x))
    d_w = cuda.to_device(np.ascontiguousarray(w))
    d_y = cuda.device_array((Ho, Wo), dtype=np.float32)
    grid = ((Wo + TILE - 1) // TILE, (Ho + TILE - 1) // TILE)
    conv2d_kernel[grid, (TILE, TILE)](d_x, d_w, d_y)
    return d_y.copy_to_host()

@cuda.jit
def conv2d_kernel(x, w, y):
    patch = cuda.shared.array((PATCH, PATCH), float32)
    wsh = cuda.shared.array((KMAX, KMAX), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    # TODO 1: cooperative load of the (TILE + K - 1)^2 patch (zeros outside the image)
    # TODO 2: load the filter into wsh
    # TODO 3: barrier, then compute y[row, col] from shared memory
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0], [8.0, 9.0, 10.0, 11.0], [12.0, 13.0, 14.0, 15.0]], "dtype": "float32"}, {"__nd__": [[1.0, 0.0], [0.0, -1.0]], "dtype": "float32"}], "e": {"__nd__": [[-5.0, -5.0, -5.0], [-5.0, -5.0, -5.0], [-5.0, -5.0, -5.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[9.0, 9.0, 9.0, 9.0], [9.0, 9.0, 9.0, 9.0], [9.0, 9.0, 9.0, 9.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8259502053260803, 0.24250569939613342, -0.06385108083486557, -0.09436370432376862, 0.3234521448612213, 0.6659548282623291, 0.18949203193187714, 0.17836442589759827, -1.137323021888733], [0.5780571699142456, -0.32551246881484985, 0.9899712204933167, 1.3131839036941528, -1.6326286792755127, 1.3297775983810425, -0.6809801459312439, -1.0375930070877075, -0.23238547146320343], [0.2991819679737091, 1.0089105367660522, 2.822406530380249, -1.463409185409546, -0.36645999550819397, 0.37620946764945984, -0.5322597622871399, -1.7193819284439087, -0.22773583233356476], [-0.02699616551399231, 0.39276322722435, -0.3918745517730713, 1.3500761985778809, 0.21866917610168457, 0.7944055199623108, -1.0297352075576782, -2.5514321327209473, 0.7805717587471008], [0.3619139790534973, 0.6996249556541443, 0.1537962257862091, -0.301779568195343, -0.2358541339635849, 1.0283503532409668, -0.7384398579597473, -0.26996272802352905, -0.01646169275045395], [0.3445723056793213, 2.217026472091675, -0.5112471580505371, 2.5386464595794678, 0.17556796967983246, 0.03642299771308899, -0.4208109676837921, 0.9349409937858582, 1.6998803615570068], [-0.8703067898750305, -0.08158627897500992, -0.22924628853797913, -0.5244739055633545, -0.9696029424667358, 0.4599975049495697, 0.7796033620834351, 0.3293728232383728, -0.47774097323417664]], "dtype": "float32"}, {"__nd__": [[2.0]], "dtype": "float32"}], "e": {"__nd__": [[-1.6519004106521606, 0.48501139879226685, -0.12770216166973114, -0.18872740864753723, 0.6469042897224426, 1.3319096565246582, 0.3789840638637543, 0.35672885179519653, -2.274646043777466], [1.1561143398284912, -0.6510249376296997, 1.9799424409866333, 2.6263678073883057, -3.2652573585510254, 2.659555196762085, -1.3619602918624878, -2.075186014175415, -0.46477094292640686], [0.5983639359474182, 2.0178210735321045, 5.644813060760498, -2.926818370819092, -0.7329199910163879, 0.7524189352989197, -1.0645195245742798, -3.4387638568878174, -0.4554716646671295], [-0.05399233102798462, 0.7855264544487, -0.7837491035461426, 2.7001523971557617, 0.43733835220336914, 1.5888110399246216, -2.0594704151153564, -5.1028642654418945, 1.5611435174942017], [0.7238279581069946, 1.3992499113082886, 0.3075924515724182, -0.603559136390686, -0.4717082679271698, 2.0567007064819336, -1.4768797159194946, -0.5399254560470581, -0.0329233855009079], [0.6891446113586426, 4.43405294418335, -1.0224943161010742, 5.0772929191589355, 0.3511359393596649, 0.07284599542617798, -0.8416219353675842, 1.8698819875717163, 3.3997607231140137], [-1.740613579750061, -0.16317255795001984, -0.45849257707595825, -1.048947811126709, -1.9392058849334717, 0.9199950098991394, 1.5592067241668701, 0.6587456464767456, -0.9554819464683533]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.2639122009277344, 1.0128015279769897, 0.4936886131763458, 1.0317952632904053, -0.14305320382118225], [0.8621152639389038, 0.7112765312194824, -0.5290629267692566, -0.08474244922399521, -0.6301223635673523], [0.19252541661262512, -0.10737854987382889, 0.11161237955093384, 0.023622574284672737, -0.5310258865356445], [0.5446661114692688, -1.074299931526184, 0.038689568638801575, -2.2590341567993164, -0.9815101623535156], [0.7269859910011292, -0.21877874433994293, 0.3005310595035553, -0.926393985748291, -1.4965473413467407]], "dtype": "float32"}, {"__nd__": [[-1.0616674423217773, 0.00032061655656434596, -0.401600182056427, 1.7527967691421509, 1.885930061340332], [0.865405797958374, -0.8129751682281494, 0.13590742647647858, -0.45002540946006775, -0.5285801291465759], [-0.31880152225494385, 0.025437641888856888, -0.26843178272247314, -0.05047791823744774, -1.8294280767440796], [-0.5421152114868164, 0.6674636602401733, 0.5399990081787109, -1.2990834712982178, 0.24659347534179688], [0.039769984781742096, -1.257469654083252, 0.11606379598379135, 0.44213128089904785, -0.05350891500711441]], "dtype": "float32"}], "e": {"__nd__": [[4.114551067352295]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.7125557661056519, -0.6014092564582825, 0.21537384390830994, 0.4494301378726959, 0.45338174700737, -0.48581141233444214, 0.4895194172859192, 0.2937462627887726, 2.3015966415405273, 1.3027598857879639, -0.030962705612182617, 1.347259283065796, -0.9470467567443848, 0.25213542580604553, 1.6000816822052002, -1.8596808910369873, 1.0507853031158447, 0.5109121799468994, -0.6270012259483337, 0.8402518033981323], [-1.1985042095184326, 0.9144647717475891, -0.022210657596588135, 0.9838637113571167, -1.164842963218689, 0.5795301198959351, 0.81592857837677, -0.9996020793914795, 1.602165699005127, -1.1282457113265991, 2.126728057861328, 2.002566337585449, -0.8154894709587097, 1.6624867916107178, 0.3430386781692505, 0.36445483565330505, 0.3595871925354004, -1.732729196548462, -0.02013220265507698, -0.2643895149230957], [0.554594874382019, -2.200727939605713, -0.009908976033329964, 1.0102354288101196, -0.672204315662384, 0.5590479969978333, 0.5639139413833618, -0.25254401564598083, 0.29309365153312683, 0.8469935655593872, -1.5315656661987305, 0.6347637176513672, -0.2987065017223358, 0.02044641226530075, -0.45545461773872375, -0.5637280344963074, 0.6983070373535156, 0.41612014174461365, -0.3174668252468109, -1.629256248474121], [0.3675061762332916, 0.49878817796707153, 1.9832438230514526, -0.6872155666351318, 0.18048152327537537, -0.24994565546512604, -2.6587960720062256, -0.7214406132698059, 0.4310142397880554, -0.6661023497581482, 1.656606674194336, 0.511338472366333, -0.3328644931316376, 1.0576249361038208, -0.35197851061820984, 1.0393847227096558, 0.29063326120376587, 0.5084342360496521, -0.5944845080375671, 0.07955393940210342], [0.3877643346786499, 0.21226641535758972, 0.6498911380767822, 1.7784823179244995, -0.15460023283958435, -0.7165502309799194, 0.7287430167198181, -0.48030951619148254, 1.5182569026947021, -0.40312737226486206, -0.7905418872833252, 0.35777726769447327, -1.0699145793914795, -0.4318791925907135, 0.7590444684028625, -1.785906195640564, 0.259724885225296, -0.4757724106311798, -0.000810332945547998, 0.9531633853912354], [0.35725826025009155, 0.6105040311813354, -0.5009567141532898, 1.3764005899429321, 0.4486114978790283, -1.8969025611877441, 1.2862846851348877, -1.2598223686218262, 1.18925142288208, -0.9482790231704712, 0.49208521842956543, 1.1024161577224731, -0.22833210229873657, 0.35899877548217773, 0.622451901435852, -1.7799923419952393, 1.860599160194397, -0.1775558441877365, 0.46330517530441284, 0.14123982191085815], [0.9154083728790283, 0.079976387321949, -1.216665506362915, 0.8651014566421509, -2.276805877685547, 0.8999141454696655, -1.7306501865386963, 1.1130282878875732, 0.605711817741394, 0.05746941640973091, -0.45641499757766724, 0.010759515687823296, -0.8220455646514893, 0.6460092663764954, 0.4824202060699463, 0.9649843573570251, -0.8940181732177734, -0.6259673237800598, -0.5379915237426758, -1.9847197532653809], [-1.3439568281173706, 0.34687572717666626, 1.2571011781692505, -0.482675701379776, -0.8067238926887512, 1.4664338827133179, -1.833911418914795, 0.9466080665588379, -1.731567144393921, 0.46409106254577637, -0.04615692049264908, 0.597416341304779, 0.21543462574481964, -0.40268781781196594, -0.24559001624584198, -0.5197955965995789, 1.0359970331192017, -1.02946138381958, 1.0254173278808594, -0.26142647862434387], [-1.2024471759796143, 1.3131612539291382, -1.1797465085983276, -1.1674644947052002, -0.06638150662183762, 0.1992027461528778, 1.0801680088043213, 1.0756349563598633, -0.5810515284538269, -0.12998783588409424, -0.379920095205307, -1.835439682006836, -0.3014102280139923, -1.941117763519287, -0.19542360305786133, -1.073304533958435, -0.731380820274353, 0.4359282851219177, 0.3231683671474457, 1.148436188697815], [-1.5933809280395508, 0.22684499621391296, 1.1047085523605347, -1.3324848413467407, -1.1740185022354126, -0.6756793260574341, 0.22148287296295166, -1.2189185619354248, 0.644396960735321, -0.8591756224632263, -3.3112120628356934, 0.19460910558700562, -0.1303635984659195, 0.051277149468660355, 1.4685063362121582, -0.40446218848228455, -0.5429732799530029, 0.1085219532251358, -1.0713822841644287, -0.42805713415145874], [-0.6572598814964294, 0.20273295044898987, -0.6462482810020447, -0.659571647644043, -1.3668886423110962, 1.5406346321105957, 0.29357224702835083, 0.6753513216972351, 1.1857744455337524, 0.8024131655693054, -0.14925342798233032, -0.0805872455239296, 2.678891897201538, -0.9818759560585022, -0.5037511587142944, 0.8983280062675476, 0.38517555594444275, 1.0261768102645874, -1.672303557395935, -1.0337985754013062], [0.014339049346745014, -1.2518212795257568, 1.9173506498336792, -0.4931294620037079, -0.8648459315299988, -0.16459594666957855, 0.4498123526573181, 1.6843656301498413, 0.414920836687088, -0.32762837409973145, 0.55937260389328, 0.7143830060958862, -0.10870036482810974, 1.7104387283325195, -0.29728880524635315, -1.2163575887680054, 0.7588982582092285, 0.7996290326118469, -1.738601803779602, 0.2070212960243225], [-1.3741309642791748, -0.42217934131622314, -0.2472183257341385, -0.6773029565811157, 0.2121458351612091, -0.08466950803995132, -0.5067789554595947, -0.2969818413257599, 0.5126289129257202, -0.4856659770011902, 0.057203616946935654, 0.39053651690483093, 0.8788315653800964, -1.9579887390136719, 0.5981754660606384, -1.578354835510254, 0.7006268501281738, 0.9560953378677368, 0.9636273980140686, 0.19955027103424072], [-1.080054759979248, 0.09730852395296097, -0.6957063674926758, 1.2316869497299194, 0.6463930606842041, 0.926933228969574, 0.0023696566931903362, -1.1783889532089233, 0.32056450843811035, 0.3506062924861908, 1.8554131984710693, 0.649002730846405, 0.5954604148864746, -1.1491202116012573, -0.3180794417858124, -1.361621618270874, 2.4975392818450928, 2.614577293395996, 0.9079113602638245, 1.1528819799423218], [0.3566175699234009, 0.2632226347923279, -1.249567985534668, 0.6089494824409485, -0.37067073583602905, 1.4091795682907104, 0.3340964615345001, 0.3482675552368164, 0.7798206210136414, 0.3255348205566406, -0.26275110244750977, -0.15672911703586578, -1.0660686492919922, 0.20601710677146912, -0.498180627822876, 0.16504253447055817, -0.18862313032150269, 0.45975324511528015, 0.42793670296669006, 0.18957139551639557], [0.48280733823776245, -0.88417649269104, 0.8203648328781128, -1.5629199743270874, 0.5568755269050598, 0.7788670063018799, 1.1618824005126953, 0.4099130928516388, 0.8313145041465759, -1.0973681211471558, 0.49374160170555115, -0.6547407507896423, 0.5922554731369019, 1.5989009141921997, -1.014873743057251, -0.7868112325668335, 1.1679539680480957, 0.12673112750053406, 1.3847119808197021, -0.5542989373207092], [-0.8198671936988831, 1.0531972646713257, 1.4338229894638062, -1.1093740463256836, -0.13098789751529694, 0.9842607378959656, -1.0315814018249512, -1.8678418397903442, -0.8703606128692627, 0.19741979241371155, 1.2431093454360962, 0.11739645153284073, -0.6429968476295471, -0.17426222562789917, -2.410444498062134, -0.7021455764770508, -1.5027356147766113, -0.5849263072013855, 0.9071291089057922, -1.5307079553604126], [0.39610540866851807, 0.12594352662563324, -0.11740859597921371, -0.7326738238334656, -0.14001095294952393, 0.609326183795929, 1.435193419456482, -0.9552847146987915, 0.4863234758377075, -0.7772759795188904, 0.7020941972732544, 0.3685777187347412, -2.224698781967163, -0.7024269104003906, -1.3192138671875, -0.5983452200889587, -0.32671961188316345, -0.25283828377723694, 0.40150201320648193, 0.11952104419469833], [0.42731204628944397, 0.9340022206306458, -0.8772966265678406, -0.4233040511608124, -1.5702463388442993, -0.9550529718399048, 0.7291519641876221, 0.43940258026123047, 0.5626270174980164, 1.709393858909607, 0.40526390075683594, 0.28575754165649414, 0.5493799448013306, 1.6837447881698608, -0.9043266773223877, -1.6578640937805176, 0.6876978874206543, -1.290393590927124, 0.6030962467193604, -0.7417945861816406], [-0.8181323409080505, -0.03645313158631325, -2.8142802715301514, -1.29421865940094, 0.27681636810302734, 1.4005553722381592, 1.522461175918579, 1.024205207824707, -0.605390727519989, -0.2596054971218109, 1.6424859762191772, 0.7136921286582947, 0.4032268822193146, 0.12028888612985611, 0.05051887035369873, -0.45350030064582825, -0.9092881679534912, -0.49508658051490784, 0.7137307524681091, -0.01856336183845997]], "dtype": "float32"}, {"__nd__": [[0.2603335678577423, 0.5088376402854919, 0.4399188756942749, -2.1080615520477295, 0.5954119563102722], [0.08818762004375458, 0.9568098783493042, 1.613824725151062, 0.3203573226928711, 0.6730908155441284], [-0.01066429540514946, 0.5763866901397705, 0.4223964512348175, -0.8120185732841492, 1.6611696481704712], [0.9829142689704895, 2.2323920726776123, -0.3586421608924866, 0.49990835785865784, -1.8801993131637573], [-1.155712366104126, 0.5215450525283813, -0.5180283188819885, -0.08794151246547699, -0.13623341917991638]], "dtype": "float32"}], "e": {"__nd__": [[-4.122768402099609, 7.4966206550598145, 7.820364475250244, -4.185642719268799, 3.2883801460266113, -6.5466485023498535, -11.526167869567871, 9.10417366027832, -2.9595601558685303, 9.466721534729004, 7.3846869468688965, -6.124216079711914, 11.876112937927246, -1.2814421653747559, -0.6767809391021729, 0.9858723282814026], [-1.0619717836380005, 6.164397716522217, -0.8765448927879333, 1.5400334596633911, -2.3864548206329346, -0.2166033238172531, 4.303271770477295, 2.176356315612793, -5.341914653778076, 5.875718593597412, -4.932830333709717, 2.335270881652832, -3.7561259269714355, 2.598752975463867, -2.3634190559387207, 0.7403709888458252], [0.3800916373729706, 4.078686237335205, 2.303243398666382, -1.2077300548553467, -3.8860983848571777, -4.2544732093811035, -3.556809902191162, 5.12584924697876, -2.7854840755462646, 1.8263866901397705, 3.9250941276550293, 1.4343243837356567, 1.28394615650177, 1.670763373374939, -4.9637651443481445, 4.190147399902344], [11.757170677185059, -5.344947814941406, 9.715485572814941, -8.056870460510254, 6.92535400390625, -11.513973236083984, 9.193402290344238, -1.1095672845840454, 0.09047598391771317, -0.50677090883255, -2.0910582542419434, -8.21871280670166, 5.804394721984863, -1.638603925704956, 1.5053303241729736, 7.041743278503418], [-4.683045387268066, 3.3478047847747803, 6.402573585510254, -6.914807319641113, 8.916857719421387, -10.513574600219727, 2.8852767944335938, -1.9474353790283203, -2.6509337425231934, 7.662337303161621, 3.3790454864501953, 2.365017890930176, 0.905633807182312, 1.6516637802124023, -2.4503118991851807, 1.5807548761367798], [-2.337116241455078, 2.0254437923431396, -10.175759315490723, -2.3687825202941895, 0.6783508062362671, 2.254948377609253, 7.112133979797363, 5.850661277770996, -4.007944107055664, 2.6687729358673096, -1.4420645236968994, -1.9525212049484253, 3.898963451385498, -8.136308670043945, -1.8405907154083252, -9.436385154724121], [0.7477145195007324, 9.029664039611816, -6.565409183502197, 4.191137790679932, -9.191116333007812, -0.394567608833313, 0.3041508197784424, -4.013227939605713, -0.19939348101615906, -10.841848373413086, -7.46944522857666, -0.694316565990448, -7.213296890258789, 4.883730411529541, 5.785904407501221, -2.1311140060424805], [0.2678777575492859, -1.576699137687683, -8.824049949645996, -2.8291354179382324, 0.9357779026031494, 4.546408176422119, -4.960542678833008, 2.421086311340332, -8.275972366333008, -2.9534754753112793, -0.6084212064743042, -3.2556936740875244, 0.020091019570827484, -9.816431045532227, 3.206699848175049, 2.2979323863983154], [0.4539027214050293, 4.404717445373535, -5.49577522277832, -8.490132331848145, -1.343065857887268, 1.913833498954773, 1.035557746887207, 1.27023446559906, 1.081072449684143, -10.44865894317627, 7.916669845581055, 4.660865306854248, 1.1703202724456787, 6.402148723602295, -8.094749450683594, -0.6464164853096008], [-2.1686744689941406, 0.625653088092804, -1.2334485054016113, -0.1581844687461853, 3.5529794692993164, -0.42564836144447327, 1.87715744972229, 11.347851753234863, -3.867790937423706, 5.996363162994385, -1.458126425743103, 3.642066240310669, -3.86310076713562, 3.317030906677246, -10.714632987976074, 6.147126197814941], [1.1261889934539795, -1.7135523557662964, -2.6205825805664062, 2.5809857845306396, 4.343578338623047, -0.6751499176025391, -4.3639116287231445, 1.790421724319458, 3.2801952362060547, -0.776060938835144, 10.859643936157227, 5.200266361236572, -4.493328094482422, -6.562777996063232, -7.325960159301758, 6.651192665100098], [0.5890496969223022, -1.5445410013198853, -1.9095911979675293, -1.837183952331543, -1.646200180053711, -0.1075238436460495, 4.767936706542969, 2.33410382270813, 2.152282476425171, 0.7787379026412964, -1.1459046602249146, -7.255814075469971, 7.927216053009033, -5.092435836791992, -3.4671616554260254, 11.303438186645508], [-3.6649274826049805, 3.6259257793426514, -5.15308952331543, 3.1856343746185303, 5.469618797302246, 2.0056657791137695, 2.3415868282318115, 8.025276184082031, -0.7055107951164246, -1.6984221935272217, 5.534674167633057, -3.602186918258667, 6.49051570892334, -1.6997056007385254, 0.1386469304561615, 10.043122291564941], [-2.0360946655273438, 3.526193380355835, 0.9395373463630676, 4.7918701171875, 10.684150695800781, -1.8009519577026367, -5.803708076477051, -5.679716110229492, -0.317813903093338, 5.195932865142822, 4.962960243225098, -2.550058364868164, 11.381699562072754, -10.838642120361328, -6.201573848724365, 0.46667802333831787], [1.7804502248764038, 0.1556582897901535, -8.169376373291016, 1.7832950353622437, 4.201441287994385, 7.074290752410889, -1.662397027015686, -2.4344637393951416, 2.231997489929199, 2.5234837532043457, -1.4146897792816162, -1.261378526687622, -5.6143364906311035, -7.083932399749756, -1.4007647037506104, -1.9616183042526245], [14.870868682861328, -1.0163723230361938, -2.3921313285827637, -8.175819396972656, -3.8539345264434814, -10.42883586883545, 3.0139622688293457, -2.447470188140869, 2.1090211868286133, 1.9886895418167114, -7.13983154296875, 1.8176608085632324, -1.2341511249542236, -5.052855968475342, -8.569432258605957, -4.0927252769470215]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.0434784889221191, 0.9792272448539734, 1.2252000570297241, 2.0909039974212646, -0.8135280609130859, -0.11310591548681259, -0.7077620029449463, -0.7423956990242004, 0.9810295701026917, 2.041088581085205, -0.2636983394622803, -0.07097810506820679, -0.6593176126480103, -1.0900685787200928, -1.0672448873519897, 0.7623058557510376, 0.6990886926651001, 0.1697378009557724, -0.19469016790390015, -2.067675828933716, -0.7421483993530273, 0.006030261516571045, -1.0397241115570068, 0.2714918553829193, 1.1208841800689697, -0.3102843761444092, 0.0813281387090683, -0.594383180141449, 0.3540954291820526, 0.2719484269618988, -1.6939127445220947, -0.7308160662651062, -1.4624269008636475, 0.6552428603172302, 1.1278929710388184, 0.30659806728363037, -1.5579204559326172], [-0.43707653880119324, -1.6659343242645264, 1.2358726263046265, -0.5519977807998657, 0.04543699324131012, -0.5271462798118591, 1.2509732246398926, -1.14517343044281, 1.4278041124343872, 0.3626616299152374, -0.5928534269332886, -0.8167002201080322, 1.1488597393035889, -0.21712100505828857, 0.3906579911708832, -0.176193967461586, 1.1117500066757202, 0.05105900391936302, 0.40719568729400635, 0.4009064733982086, 1.6876128911972046, 0.4556204080581665, -1.6389387845993042, 2.2589781284332275, 0.8159598708152771, -0.418178528547287, -0.4474736154079437, 1.2603005170822144, 0.1088857427239418, -0.051325175911188126, -2.5295286178588867, 0.31102871894836426, -0.14978763461112976, 1.9818906784057617, -0.021651824936270714, -0.5450679063796997, 0.3688015341758728], [0.7969503998756409, 0.5401269793510437, 0.28781163692474365, 0.8371517062187195, -1.2023543119430542, -1.1398276090621948, 0.705996572971344, 0.7039037942886353, 0.01915479451417923, -2.0314300060272217, -1.0499848127365112, -3.237018585205078, 2.5954856872558594, 0.6291397213935852, 0.9538556933403015, 0.17497679591178894, -1.0434118509292603, 0.7648144960403442, -0.7380685210227966, -0.05641800910234451, -1.4526865482330322, -1.2788748741149902, 1.197603702545166, 0.8322741389274597, -1.3212275505065918, 0.6359195113182068, -0.3011462986469269, 0.3152795732021332, 2.02329158782959, -0.05719561502337456, 0.09530207514762878, 0.6218334436416626, -0.3596012592315674, 1.2635364532470703, -0.9759529829025269, -0.7884812355041504, -0.7747665643692017], [0.13591648638248444, 2.685656785964966, -0.5201758146286011, -0.42273855209350586, 0.9136853814125061, -0.5824644565582275, -0.40514492988586426, 0.023954546079039574, -1.460587978363037, -0.07608585059642792, 0.3923934996128082, 0.41318848729133606, -0.9684256315231323, 2.3125524520874023, -1.551559567451477, -1.1391675472259521, -1.4315294027328491, -1.6900798082351685, -0.5888922810554504, -1.5263432264328003, 0.6300870180130005, -0.3537355363368988, 0.8935701251029968, -0.4169715940952301, -0.2985616624355316, 0.1730486899614334, 0.6154730319976807, -0.4717550575733185, 0.14678741991519928, 0.6159552931785583, -1.128126859664917, -0.5575012564659119, 0.24152430891990662, 0.12637372314929962, 0.1035851314663887, 0.5454204678535461, 0.0015572684351354837], [1.4188112020492554, 0.922761082649231, -0.23731744289398193, 0.2797703444957733, 0.9653293490409851, -0.07656823843717575, -0.3909144103527069, -0.7048698663711548, -1.8550939559936523, 0.24634553492069244, 0.974083423614502, -0.07348790764808655, -2.192676067352295, 0.2688274681568146, 0.0968225970864296, -2.210634469985962, -0.6976023316383362, -0.8970103859901428, 0.3974681794643402, 0.48740869760513306, 1.370484471321106, 0.3072141110897064, 1.6774799823760986, -0.5752806663513184, 0.03889252245426178, 0.0009134990395978093, 0.18584652245044708, -0.9795522093772888, -0.35187146067619324, 0.8279240131378174, -0.5008133053779602, -1.0928499698638916, -0.9629387855529785, 1.5855977535247803, -0.0251106359064579, -0.8468988537788391, 0.8401840925216675], [-0.2765377163887024, 1.3588107824325562, -0.6124110817909241, -0.2477637529373169, -0.4618994891643524, 0.27029484510421753, -2.013049840927124, 1.26055908203125, 2.4632503986358643, 0.16270974278450012, 0.30381765961647034, -1.3436741828918457, 0.9207574725151062, -1.4461004734039307, 0.1847490817308426, 0.8448923826217651, -0.870138943195343, -1.3022342920303345, 0.2796955108642578, 0.3567969501018524, 0.02205692231655121, 0.812981903553009, 0.9900043606758118, -0.26497671008110046, 0.007395839784294367, 1.5830351114273071, 0.28853607177734375, -0.1027861088514328, -0.34540703892707825, -0.3120361864566803, -0.5487902164459229, -0.0994795486330986, -0.14577586948871613, 0.9607657790184021, -0.7379117608070374, -0.6783350706100464, -0.665637731552124], [0.48242971301078796, -0.03359915316104889, -0.558229386806488, -0.36116158962249756, -0.42663469910621643, 0.3163394629955292, -0.7578281760215759, 1.1026780605316162, 0.42397984862327576, 0.0038940305821597576, -0.5011886954307556, 1.3155572414398193, 0.6750196814537048, -1.274593472480774, -1.3348338603973389, 1.0868380069732666, -0.35572150349617004, 1.3962785005569458, 1.8401473760604858, -0.112474724650383, -1.3579777479171753, 2.1236934661865234, -1.5883692502975464, -0.8344048857688904, 0.5074043273925781, -0.3749825060367584, -2.7911739349365234, -0.645568311214447, 0.09809450060129166, 0.7973930239677429, 0.7576674818992615, 0.10602808743715286, -1.039673089981079, -1.3802709579467773, 1.799311876296997, 0.15040238201618195, 1.0135385990142822], [-0.7251518964767456, -0.8415220379829407, 1.2314152717590332, -0.7888529896736145, -1.029024362564087, 0.41784608364105225, -1.7242052555084229, -1.4313417673110962, -0.1714034229516983, 0.5496102571487427, 2.7807114124298096, -0.0973397046327591, 0.0297283586114645, -0.5915456414222717, -0.4084298014640808, -1.1887125968933105, 0.028027517721056938, 1.0765231847763062, -0.3467722535133362, 1.96591055393219, 0.2768443524837494, -0.36291348934173584, -0.27899131178855896, -1.2571933269500732, 0.4224039912223816, -0.9465740919113159, -0.4835248589515686, 1.010891079902649, 0.8975194692611694, 0.48663365840911865, -1.9219852685928345, -0.22728894650936127, -0.09819824248552322, 0.840565025806427, -0.42327573895454407, 0.3055865168571472, -1.9669408798217773], [-1.913589596748352, -0.002957464661449194, 1.4456098079681396, 0.022241810336709023, -1.1452172994613647, 0.5812764763832092, -1.8928413391113281, 1.2009000778198242, 1.5527865886688232, 0.5709195137023926, -0.43121209740638733, -2.043701410293579, 1.2732036113739014, 0.22709433734416962, 0.5347557067871094, 0.0437542088329792, -0.6923345327377319, -0.13722553849220276, -0.577167272567749, -0.584641695022583, -1.635198950767517, -0.7233151793479919, 1.3382084369659424, -1.4201217889785767, -1.99657142162323, -0.08619705587625504, -0.24029579758644104, 0.4902777373790741, -2.462552070617676, -0.916229248046875, -1.2340940237045288, 0.6575785875320435, -1.0218228101730347, 0.43221545219421387, -0.16313612461090088, -0.6200351119041443, 0.17737917602062225], [-0.5333237648010254, -0.8973413109779358, -0.48231038451194763, -0.4569803476333618, 0.7368785738945007, -0.831360399723053, -0.36218971014022827, -0.03373565152287483, -0.10952503234148026, 1.2475090026855469, -1.8096023797988892, 2.0586252212524414, 1.8788269758224487, -1.508885145187378, -1.0666636228561401, -0.07752573490142822, 1.1063835620880127, 1.6496894359588623, -0.5153011083602905, -0.9572575688362122, 0.9088637828826904, -0.190651997923851, -1.1393924951553345, -3.0188944339752197, -1.705344557762146, -1.5996928215026855, -0.7896014451980591, 0.3874244689941406, -1.0121818780899048, -1.057478427886963, 0.8740781545639038, 1.0264984369277954, -0.3604946732521057, 1.2186399698257446, 1.2907261848449707, 0.790727972984314, 0.9231674075126648], [0.06320517510175705, -1.3138870000839233, -0.7178650498390198, -0.3908992111682892, -2.303225040435791, -0.1924743801355362, -0.4196276366710663, -0.19732558727264404, -0.3694201111793518, -0.6558701992034912, -0.22740721702575684, -1.9511528015136719, -0.9259158968925476, 0.08430074900388718, -0.08895561844110489, 0.7429877519607544, 0.28402888774871826, 0.10814634710550308, 0.3195209503173828, 0.09140536189079285, 0.0728154107928276, -1.1482487916946411, -0.10733121633529663, -0.5005882382392883, 0.406606525182724, -0.7432829141616821, -0.694874107837677, 0.6366550326347351, -1.0549367666244507, -0.1508668214082718, -1.1377298831939697, 0.10291653126478195, 0.1095517948269844, -0.528344988822937, -0.49727895855903625, 0.18656210601329803, -0.8172829151153564], [-1.513067364692688, 0.9591399431228638, 0.12094246596097946, -1.2613804340362549, -0.7621649503707886, -0.37690478563308716, -1.4388794898986816, 0.9755547046661377, -0.5025084018707275, -0.7280367612838745, 0.6037013530731201, -1.6933377981185913, -1.6077347993850708, 0.25256067514419556, 0.1613815426826477, -0.9732798337936401, -0.5447815656661987, 2.428701877593994, 1.283194899559021, 0.21720898151397705, 1.420702576637268, -1.158201813697815, 0.8822774887084961, -0.7036603689193726, -0.032379310578107834, 0.8868815898895264, -0.8455820083618164, 0.7967743277549744, -0.015873489901423454, 1.4618867635726929, 0.8131070733070374, 0.08338669687509537, 0.3276449739933014, 1.9107534885406494, 0.14386771619319916, 0.3252883553504944, 0.033003948628902435], [0.867069661617279, 0.8371415138244629, -0.0740862786769867, 0.08250157535076141, -0.7981072068214417, -0.41588038206100464, -0.458896666765213, -0.37154436111450195, -0.5942202210426331, -0.7634654641151428, -1.9391565322875977, 1.4889999628067017, 0.23538263142108917, -0.3476422429084778, 0.31307587027549744, 0.0017746767261996865, 0.11540557444095612, 0.9887538552284241, -0.5698935389518738, 0.686519980430603, -0.4603264331817627, -0.42373955249786377, 0.3606492578983307, -0.3939706087112427, 1.3831369876861572, 1.3772644996643066, 1.4211348295211792, 0.47595879435539246, -0.4883771538734436, 0.8192426562309265, 1.135159969329834, -1.198509693145752, 0.06304684281349182, -0.28980669379234314, -1.5660687685012817, 0.6538504362106323, 0.9609819054603577], [1.8777685165405273, -0.5349611639976501, -2.42683744430542, 0.8838366866111755, -1.0335274934768677, 0.9778746962547302, 0.7829833030700684, 0.719504714012146, -2.174518585205078, -0.16011135280132294, -0.1603507101535797, -1.2647985219955444, 1.5848079919815063, -0.36504998803138733, 0.060350172221660614, -1.1309446096420288, 1.16214120388031, -0.769922137260437, -0.5271165370941162, 1.4417524337768555, -1.2468539476394653, 0.5405136346817017, 0.49391236901283264, 1.7954663038253784, -0.6058269143104553, 0.8018272519111633, 0.7564443945884705, 1.4665257930755615, -0.1602402925491333, -1.3651320934295654, -0.4716815948486328, -1.1370258331298828, 0.20832209289073944, -0.06403312087059021, 0.533665120601654, -1.3509633541107178, 1.5883190631866455], [-0.07655122876167297, 0.6940296292304993, -0.09886076301336288, -1.3816105127334595, 1.0908615589141846, -0.5351279377937317, -0.928198516368866, 0.6246181130409241, 0.9227699637413025, -0.7797695994377136, -0.39268991351127625, -1.0121347904205322, 0.4556809365749359, -0.523211658000946, 1.2302292585372925, 1.917837381362915, 0.543067216873169, 0.9159520864486694, 0.9708525538444519, -0.32470279932022095, -0.6469762921333313, -0.25141462683677673, 0.1667650192975998, 1.343803882598877, -2.153409004211426, 0.4024563729763031, -0.2260897010564804, 0.21734018623828888, -1.448595404624939, -0.826409101486206, -0.10657949000597, 0.006862034555524588, 0.42211300134658813, 1.5199847221374512, -1.4960938692092896, 0.04570356756448746, -0.6957844495773315], [0.6488787531852722, -0.7003469467163086, -0.8603219389915466, -0.4352674186229706, 0.15305685997009277, -1.5437067747116089, -2.000537633895874, -0.5466208457946777, 0.09912058711051941, -2.2460358142852783, 0.6590938568115234, -0.18084998428821564, 0.6759952902793884, -0.5394925475120544, -1.4428790807724, 0.2990988492965698, 0.5046248435974121, 1.8917043209075928, 1.0932291746139526, 1.472519874572754, 1.0326555967330933, -0.22473067045211792, 0.8777005672454834, 1.1218695640563965, 0.7846866846084595, 0.16852110624313354, -0.15131215751171112, -0.10844337940216064, -0.257640540599823, -0.7135087251663208, -2.0869340896606445, 1.3404016494750977, -0.15126146376132965, -0.44933876395225525, 0.6981647610664368, -2.078756809234619, 0.5342221856117249], [0.7455665469169617, 1.037421464920044, -1.1117781400680542, -1.034338116645813, 1.0285210609436035, -0.264492928981781, -1.1878989934921265, 0.8877288103103638, -2.239478588104248, -1.4773908853530884, 1.244392991065979, 0.08624337613582611, -0.21319015324115753, 0.269490122795105, -0.2850041687488556, -0.4148757755756378, -0.5599353313446045, 2.1042990684509277, 2.1067192554473877, -1.5730137825012207, 0.547730565071106, 1.2014849185943604, -0.3272283673286438, 1.565032958984375, -0.13396599888801575, 0.7001118063926697, 0.6662782430648804, 2.2735040187835693, 0.7412097454071045, -1.9462997913360596, 0.5187622904777527, -0.9184243679046631, 0.42238959670066833, -0.32820215821266174, 0.4091257154941559, 1.2754943370819092, 0.7592146992683411], [0.9780621528625488, -0.7989737391471863, 1.0879358053207397, -0.05701705440878868, -0.6011710166931152, 0.1779518723487854, 0.13708244264125824, -0.9791780114173889, 1.2940988540649414, 0.09362160414457321, 0.2330985963344574, -1.0283557176589966, -1.6254959106445312, -0.22736284136772156, -0.7534908652305603, -0.3639155924320221, 2.7895617485046387, -0.6209668517112732, -0.015339699573814869, -0.5544604659080505, -0.7579267621040344, 0.26365336775779724, -0.7835760116577148, 0.2176562249660492, -0.8795512318611145, -1.1921626329421997, -2.1087872982025146, 1.527057409286499, -1.1009666919708252, -1.3346316814422607, 1.2440009117126465, 0.9971583485603333, -0.41818326711654663, -0.738286018371582, -0.49516913294792175, 0.4771782159805298, 0.7039983868598938], [-0.8171606063842773, 0.7412989139556885, -0.6402621269226074, -1.9713099002838135, -1.5459810495376587, -0.1660754382610321, 1.0659658908843994, -0.7246228456497192, 0.06452487409114838, -0.7049098014831543, -0.40521520376205444, -0.0362614281475544, 0.602610170841217, 0.8680478930473328, 2.1759705543518066, -0.7746121883392334, -1.6828023195266724, -0.17280764877796173, -1.5102512836456299, 0.8764219880104065, -0.5515623092651367, -2.0404858589172363, 0.45253878831863403, 2.057520627975464, -0.2302689105272293, 1.4174379110336304, 0.004478083923459053, -1.3362011909484863, 1.280949592590332, -0.895687997341156, 0.015755387023091316, -0.20665250718593597, -0.36238789558410645, 0.9949628710746765, 0.8596292734146118, -0.0799507275223732, -1.6187139749526978], [-0.45868194103240967, -0.1388227641582489, 0.11785070598125458, 1.1160048246383667, 1.5464991331100464, -0.1772945523262024, -0.3015201985836029, -0.5241458415985107, 0.1852796971797943, 0.1768442988395691, -0.8800647258758545, 0.06698042154312134, -0.3491588830947876, 2.522921323776245, -1.7910369634628296, 0.8364007472991943, -0.24528934061527252, 0.8492260575294495, 0.2211247831583023, -0.6859225034713745, -0.1446826159954071, -0.0023025551345199347, -0.8737453818321228, -0.4877486228942871, 0.47048696875572205, 0.41908127069473267, -1.052783489227295, -2.157883882522583, 0.0003966620424762368, -0.9831259250640869, 0.8536462783813477, 0.8852863311767578, 0.9698522686958313, -0.08893958479166031, -0.6696422100067139, -2.471092700958252, 0.9672458171844482], [0.9471096992492676, 1.2647727727890015, 0.6847537159919739, 0.9026342630386353, -0.608354926109314, 0.7660155892372131, 0.9518554210662842, 1.075941562652588, -0.7204955816268921, 1.5257322788238525, 0.9143217206001282, -0.5822712182998657, 0.7788839936256409, 0.8571491837501526, -0.014506981708109379, 1.5391254425048828, 1.7791633605957031, -0.383788526058197, 0.40242257714271545, 1.5329763889312744, 0.17934958636760712, -0.29095935821533203, -0.421944797039032, 1.0463069677352905, 0.6604810953140259, 0.1392802894115448, -0.1031128540635109, -0.7722125053405762, 0.7839697003364563, -0.45895838737487793, -2.059680700302124, 0.5780761241912842, -0.015770578756928444, 0.40256616473197937, 0.8761458396911621, 1.6491702795028687, -1.6973670721054077]], "dtype": "float32"}, {"__nd__": [[-0.1165366843342781, 0.006500953808426857, 1.149436593055725], [0.4996647238731384, 0.808781087398529, -0.34465673565864563], [-0.9236825704574585, 0.76849764585495, 0.16923591494560242]], "dtype": "float32"}], "e": {"__nd__": [[-0.7277954816818237, 2.518554925918579, -0.7348494529724121, -2.3265132904052734, -1.2001370191574097, 2.0128233432769775, 0.3041290044784546, 1.91732656955719, -0.9496304988861084, 0.18370682001113892, -3.1588354110717773, 4.438027858734131, -2.64566969871521, 1.446035385131836, -0.3207628130912781, 0.07028910517692566, 1.5789368152618408, -3.4642369747161865, -0.5052089095115662, 0.3705570101737976, 1.2297252416610718, 0.6706241965293884, 1.4479846954345703, -0.12470835447311401, 1.8426892757415771, -2.4172072410583496, 2.0147294998168945, 2.373635768890381, -2.998533010482788, -2.8301029205322266, -2.120060682296753, -0.4568811357021332, 4.144677639007568, -0.6065454483032227, -2.334541082382202], [4.046864032745361, -3.1701712608337402, 1.450127363204956, 0.29147255420684814, -1.6969220638275146, -1.2571667432785034, 2.5488555431365967, 0.46948593854904175, -0.7599532604217529, -1.2916851043701172, -2.861715316772461, -0.6165249347686768, 4.199826717376709, -2.670391321182251, 2.5243942737579346, -1.267530918121338, 0.6142042875289917, 1.1123961210250854, 1.4588100910186768, 1.5598472356796265, -4.953132152557373, 3.5184216499328613, 1.6741299629211426, -1.4255272150039673, -0.14128562808036804, 1.6932913064956665, -1.3135262727737427, 2.2612881660461426, -1.8418376445770264, -1.3490338325500488, 1.4535859823226929, 2.547172784805298, 1.0756220817565918, -0.6886765956878662, -0.11169757694005966], [2.0192079544067383, 0.9807296395301819, -1.7292752265930176, -0.21656434237957, 0.05263550579547882, -0.029433831572532654, -0.22990095615386963, -4.292878150939941, -0.08122771978378296, -2.845304012298584, 2.621384382247925, -1.8282562494277954, 4.9675493240356445, -0.12076368927955627, -4.418928146362305, 1.0614453554153442, -2.578740119934082, 0.2630234360694885, -3.0906424522399902, -0.9489999413490295, 0.5124000310897827, 2.711146116256714, -3.425652265548706, 0.6773301362991333, -0.41316258907318115, 1.0093744993209839, 1.2541859149932861, 0.3558726906776428, 1.710237741470337, -1.0250091552734375, -2.0584511756896973, 1.7885422706604004, 1.2192416191101074, -2.728174924850464, -0.7750459313392639], [2.136779546737671, -2.3972384929656982, 1.1801789999008179, 0.25224965810775757, 0.2728816568851471, -1.6020644903182983, 1.4875019788742065, -1.2810287475585938, -2.5414633750915527, 1.2783833742141724, -1.1306020021438599, 2.4049136638641357, -4.497625350952148, 1.0062605142593384, -2.6397624015808105, -4.848310470581055, -1.8817954063415527, -0.37785571813583374, 0.9229915142059326, 0.8464233875274658, 2.078416585922241, 1.2412775754928589, -1.2073572874069214, 0.5075977444648743, 1.9580423831939697, -1.327920913696289, -0.8881386518478394, -0.5187621712684631, -0.6570394039154053, -0.48499056696891785, 0.00901658646762371, -1.5170989036560059, 1.6497647762298584, 0.10739541798830032, -0.9461659789085388], [0.17384451627731323, 0.0224522165954113, 0.9577427506446838, -0.6456047892570496, 0.6282296776771545, -3.4193148612976074, -1.3074976205825806, 2.2272119522094727, 2.121669054031372, 0.5171635150909424, -2.2984330654144287, -0.037090644240379333, -2.2335453033447266, -3.1005895137786865, 2.2560348510742188, -1.6517008543014526, 0.660959780216217, 0.22531937062740326, -0.06311079859733582, -0.35915660858154297, 4.715360641479492, -2.7113919258117676, 1.0350714921951294, 0.49352630972862244, 0.16445010900497437, -1.9729554653167725, 1.846248745918274, 1.6469736099243164, -0.11466634273529053, -2.0577080249786377, -2.154315948486328, 0.3143855333328247, 1.2562999725341797, 1.6417970657348633, -1.0996394157409668], [-0.02508893609046936, 0.7992185354232788, -2.802968740463257, -0.28942322731018066, -0.9746138453483582, -1.3837915658950806, 3.9048049449920654, 2.2320668697357178, 1.5022459030151367, -0.8054450154304504, -1.0431360006332397, 0.15559658408164978, -0.6890517473220825, -0.9188185930252075, -1.2130807638168335, -0.5251178741455078, 1.4744833707809448, 1.8602479696273804, 3.169560432434082, -2.6579582691192627, 2.145343542098999, -0.42075487971305847, -2.3889412879943848, 3.2988922595977783, 0.053896620869636536, -1.8492413759231567, -1.006516933441162, -1.0286356210708618, -0.9400950074195862, -1.0718780755996704, 2.3024256229400635, 1.079546570777893, -2.4158759117126465, -1.2327107191085815, 0.2807859182357788], [-0.15551821887493134, 1.5499478578567505, -1.607741355895996, -1.7697720527648926, 0.7829276323318481, -1.2550569772720337, 1.5567905902862549, -0.9838335514068604, -2.2934975624084473, 2.860992431640625, 1.186497449874878, 1.4699175357818604, -2.855081081390381, 1.381503939628601, -1.9988901615142822, -0.062051981687545776, 3.606220245361328, -1.1159051656723022, -0.6476907730102539, 2.937579393386841, -0.5316897034645081, 0.2656005918979645, -3.2042043209075928, -0.5279872417449951, -1.9204181432724, -1.9510072469711304, 0.8824659585952759, -0.4446806013584137, 3.730750322341919, -1.1899096965789795, -0.9206458926200867, -3.407860279083252, 4.205421447753906, -0.3117634952068329, 1.3740757703781128], [-0.2409215271472931, 0.740242063999176, 0.022603832185268402, 0.2977689504623413, -2.6897835731506348, -2.875627279281616, -0.22500145435333252, 2.614686965942383, 5.359665393829346, -1.7115548849105835, 0.9738084077835083, -1.451098918914795, -2.916769027709961, -0.2082948386669159, 1.726209282875061, 2.0863661766052246, -0.494228720664978, -0.28368669748306274, 0.06788896024227142, -0.4593978226184845, -3.397425413131714, -1.4046516418457031, -0.8380959033966064, -2.0262160301208496, -1.383463740348816, 1.7990963459014893, 3.0755350589752197, -2.2982029914855957, -3.586580753326416, -0.19079579412937164, 0.2974957227706909, -0.6737632751464844, 0.9135462045669556, 0.5483677983283997, -3.281996488571167], [-0.1309722363948822, -0.049861252307891846, -2.3763513565063477, -0.12925589084625244, -0.3093109428882599, 0.42551448941230774, 2.0161521434783936, -0.22164824604988098, 0.7037383317947388, -3.8673477172851562, 0.16756272315979004, 4.680651664733887, 1.4588106870651245, -1.583099126815796, -1.1336078643798828, -0.3295302391052246, 1.355865716934204, 0.2388094663619995, -3.373600959777832, -0.6742255091667175, 1.448825478553772, -0.6222302913665771, -5.100043773651123, -1.6338310241699219, -2.9822356700897217, -0.7391338348388672, -1.5791183710098267, -2.8111915588378906, -2.1338272094726562, -0.038212377578020096, 1.5133095979690552, 0.11457381397485733, -0.2963763475418091, 0.7536143660545349, 1.6495338678359985], [0.8734429478645325, -2.532672643661499, -0.1908564418554306, -2.3737292289733887, -1.4986368417739868, -0.9043338298797607, 1.6678907871246338, -0.14466388523578644, -2.6889820098876953, 3.2197988033294678, -1.1197402477264404, -3.34359073638916, -0.11219072341918945, -0.48008251190185547, 0.8648679852485657, 3.3674910068511963, 1.9955646991729736, -2.235408067703247, 0.5290874242782593, 1.0938022136688232, -4.325161457061768, -2.3144781589508057, -3.8068509101867676, -0.3879455626010895, -0.3093825578689575, -1.8603930473327637, 0.8530809879302979, -2.251096725463867, 2.134497880935669, -0.43355223536491394, -1.6640801429748535, 2.099045515060425, 2.522472381591797, -1.5545358657836914, 1.2226238250732422], [-1.0331172943115234, -0.10528293997049332, -3.2665932178497314, -2.067375898361206, -0.06498457491397858, -1.9268051385879517, -0.09597941488027573, -0.6440069079399109, -1.6366063356399536, -1.9927089214324951, 1.4110538959503174, -3.16849946975708, -1.080078363418579, 1.997789740562439, -0.4453892409801483, -1.4700403213500977, 2.141500949859619, 1.035964012145996, 1.350327968597412, -0.7328649163246155, -0.5096728801727295, 0.5369839072227478, -0.04229681193828583, 0.183095782995224, 0.16297142207622528, 0.19906887412071228, -1.9296414852142334, -1.0493371486663818, 0.9811830520629883, 1.400636911392212, -1.3389328718185425, 0.13597211241722107, 0.5257306694984436, 0.40658488869667053, 1.5550124645233154], [-1.0988649129867554, -2.452237367630005, 2.152369499206543, -2.197277307510376, -0.3058565855026245, 0.5252560377120972, -1.2668647766113281, -3.7196977138519287, 2.3600828647613525, -4.510026931762695, -2.3308053016662598, 3.8557918071746826, -1.6311442852020264, -0.8760405778884888, -1.2614295482635498, 4.462401866912842, 0.8536540865898132, 0.3215617835521698, 3.2978827953338623, -3.4290366172790527, 1.7946754693984985, -0.2681962847709656, 0.06131584197282791, -0.4399951696395874, 1.6566141843795776, 2.5693860054016113, 1.7500240802764893, -0.560916543006897, -0.007624097168445587, 2.3773245811462402, -0.5443316698074341, 2.9402334690093994, 0.32515668869018555, -1.2439711093902588, -1.8239604234695435], [1.7485917806625366, -3.4887659549713135, -1.835601568222046, 0.799793541431427, -2.008497476577759, 0.3789826035499573, 2.5842020511627197, -2.1819019317626953, -4.843466758728027, 2.261296272277832, -1.4810460805892944, 1.400553584098816, 0.19120465219020844, 2.0540599822998047, -0.7594749331474304, 0.5781219005584717, -0.1558324694633484, -0.7926566004753113, -0.3816068470478058, -1.2842156887054443, 0.541963517665863, 0.237410306930542, 3.967207908630371, -1.059251070022583, 3.8264410495758057, 0.3941064774990082, 1.0261222124099731, 0.5028462409973145, 1.0303547382354736, -1.454819917678833, -1.1195424795150757, 0.004681050777435303, -1.4159435033798218, -0.90559983253479, 1.217225193977356], [-3.7378315925598145, 1.7175666093826294, -1.9561383724212646, 1.649070382118225, -0.20730969309806824, -0.7192371487617493, -1.4181278944015503, 1.2461165189743042, -1.6722736358642578, 0.7559560537338257, 0.026864781975746155, 0.37649011611938477, -2.0203661918640137, -1.744365930557251, 4.948083400726318, 0.7678588628768921, 1.1038917303085327, 2.440300464630127, -0.6203127503395081, -0.7582696676254272, -0.8455607891082764, 2.622049570083618, 1.3543449640274048, -0.905122697353363, -0.34816235303878784, 1.2501864433288574, 0.3120979070663452, -2.737830638885498, -2.5513675212860107, -2.370313882827759, 3.0260026454925537, -1.54918372631073, 2.4571659564971924, -1.4084126949310303, -0.8680692911148071], [-0.12545472383499146, -4.553023338317871, 0.8278610706329346, 1.6924095153808594, -3.034379005432129, -1.9450039863586426, 1.0974971055984497, -3.172977924346924, -1.4143716096878052, 0.7336119413375854, -0.6061947345733643, -0.03606807813048363, 2.1120927333831787, 0.19561803340911865, -0.30998921394348145, 1.0474798679351807, 4.9548139572143555, 0.25732529163360596, -2.5397281646728516, 3.4702913761138916, 0.6591964960098267, 0.689513087272644, 0.07185741513967514, -0.0009494125843048096, 1.348753809928894, 0.450760155916214, -0.4546303451061249, -2.8610565662384033, -2.03817081451416, -0.36186426877975464, -0.5223804712295532, 3.568746566772461, -3.011737108230591, 1.755700707435608, -1.41084885597229], [-0.8076052665710449, 1.1158052682876587, -2.623661994934082, -1.6957051753997803, -0.9027010798454285, -2.0850183963775635, 0.5796850919723511, -1.461012840270996, -3.095268487930298, 0.21529603004455566, 0.1833476722240448, -1.1546485424041748, -0.3319235146045685, 0.013791725039482117, 1.3535743951797485, 3.1322181224823, -1.1503465175628662, 5.2446441650390625, 0.12059329450130463, -1.2059881687164307, 3.0153510570526123, 0.30882441997528076, 2.697585344314575, -0.5783240795135498, -0.4552416503429413, -0.301140695810318, 4.572073936462402, -0.8853729963302612, -3.5541694164276123, 3.7314414978027344, -1.0046433210372925, -2.045217514038086, 0.35690125823020935, -2.2230401039123535, 2.436763048171997], [-0.6743881702423096, -2.3270785808563232, 0.8245612382888794, -0.14815481007099152, -0.20993369817733765, 2.430715799331665, -5.1299638748168945, -0.6914517879486084, 1.6538246870040894, 1.202695369720459, -0.09610403329133987, -0.8083125948905945, -0.5586674213409424, -0.3683847188949585, -5.135340213775635, 4.1448163986206055, 4.563363075256348, -3.023878335952759, 2.1541450023651123, -0.9917393326759338, -1.309609055519104, 3.660484790802002, 1.1062049865722656, -1.4078185558319092, 1.411887288093567, -1.823502540588379, 0.5357106924057007, -0.09766082465648651, -3.4298973083496094, -0.025437941774725914, 1.7565864324569702, 0.22778040170669556, 1.0283448696136475, 0.3012281358242035, -0.40025168657302856], [1.8801878690719604, 0.9743033647537231, -1.1890912055969238, -1.8431921005249023, -2.6611530780792236, -0.2731107473373413, 1.2965291738510132, 0.8196648955345154, -0.46465983986854553, -2.68721604347229, -1.5364627838134766, 0.11478984355926514, 1.5333260297775269, -1.5013453960418701, 6.588231563568115, -3.159076690673828, 0.10978265106678009, -2.905488967895508, -1.4845563173294067, 1.5798285007476807, -2.908501625061035, -1.9002184867858887, 1.5631601810455322, -0.16507774591445923, -1.590057611465454, 1.491522192955017, -3.215581178665161, 0.7849943041801453, 0.8482975363731384, 2.6601576805114746, -0.597553014755249, -1.7942534685134888, -1.2753866910934448, 1.0005955696105957, 0.6759740114212036], [-0.8049437403678894, -3.2043933868408203, -1.328508734703064, 0.7266700863838196, 3.4492998123168945, -0.7523941993713379, -0.8674912452697754, -2.1877002716064453, 2.0538220405578613, -1.4138481616973877, -0.6857779622077942, 1.168906807899475, 4.856698989868164, -1.9960013628005981, -0.8294905424118042, -0.31178876757621765, -2.9231081008911133, 2.7804839611053467, -0.009965263307094574, -4.237478733062744, 0.3374921679496765, 2.187572717666626, 0.008345536887645721, 0.9450463056564331, 0.45758765935897827, -1.9379537105560303, -1.1737933158874512, -0.36646366119384766, -2.6515605449676514, -1.3000879287719727, 2.7326245307922363, 1.944821834564209, 2.1524674892425537, 0.6439567804336548, -4.456996917724609]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.9384385347366333, 2.394371509552002, -0.40730613470077515, -0.1906702220439911, 1.675365924835205, -1.5312501192092896, -1.921197533607483, 0.9625476002693176, -0.47310754656791687, -0.04916791245341301, -0.13386818766593933, -0.16347533464431763, -0.22968101501464844, 1.2073312997817993, 0.5913374423980713, -2.3880856037139893, 0.7941449880599976, -1.0658625364303589, 0.2107488065958023, 1.139594554901123, 0.8959685564041138, -0.7694911956787109, 0.21390318870544434, -2.5632433891296387, -1.8478549718856812, -0.9121295809745789, 1.0900849103927612, 0.24719415605068207, -0.3257962465286255, 0.39523616433143616, -0.4518032670021057, 0.15349137783050537, -1.700731635093689, -0.14465415477752686], [-0.3109254539012909, 0.6655468344688416, -1.2221248149871826, -1.2034722566604614, -0.848510205745697, -0.6074181795120239, 0.9766603112220764, -0.09447818994522095, -0.14460641145706177, -0.4643976390361786, 1.6819934844970703, -0.13618919253349304, 0.36248621344566345, 0.2158268392086029, 1.294283151626587, 1.5552644729614258, -0.19283291697502136, 0.737603485584259, -0.16858063638210297, -0.3682785630226135, 0.5711736679077148, 1.1877437829971313, -0.20495186746120453, -0.14395946264266968, -0.27619698643684387, 0.17035923898220062, -1.1664520502090454, 0.28868594765663147, 0.6493948698043823, -0.2245023548603058, 0.0004445292870514095, 1.5328776836395264, 0.03344547003507614, -0.18165665864944458], [-1.4998475313186646, -0.3952964246273041, -1.5396814346313477, 1.6795464754104614, -0.1525769680738449, 1.0232927799224854, -1.038106083869934, -0.7448210120201111, -0.24071480333805084, 0.03361253812909126, 0.501514196395874, 0.29963621497154236, 0.5888628363609314, -2.307238817214966, 0.22671706974506378, 0.6776473522186279, 0.8318747282028198, -0.11658904701471329, 1.7518973350524902, -0.9078136086463928, -0.8012749552726746, 2.233154773712158, -0.0915532335639, -1.8036199808120728, -0.7349549531936646, 0.11093773692846298, 0.07433144748210907, 1.9114118814468384, 0.9009736776351929, 0.6349818706512451, 0.9545884132385254, -0.746299147605896, 0.013468577526509762, -0.2842431962490082], [-0.6465285420417786, 1.5903799533843994, -1.6368778944015503, -0.9390184879302979, 1.3756519556045532, 1.414125680923462, 0.9861242175102234, -1.479798436164856, -0.08758589625358582, 2.7333879470825195, -1.313281536102295, -0.012686148285865784, 0.5023463368415833, -0.35738179087638855, -1.6857616901397705, -0.21292133629322052, 0.6803597211837769, -0.9605992436408997, -2.3147194385528564, 0.5533169507980347, -0.4215172529220581, 0.18240094184875488, 0.711621880531311, -1.2485554218292236, -2.784303903579712, 0.1983075886964798, 0.23878024518489838, -0.29477229714393616, 0.5489335060119629, 0.5704407095909119, -0.635307252407074, -0.1149689182639122, -0.027024080976843834, -0.6565996408462524], [0.37964269518852234, 0.8938385248184204, 0.5543954372406006, 0.3384800851345062, 1.1521257162094116, -0.019677409902215004, -0.22263865172863007, -1.642413854598999, 1.5968964099884033, 0.08138296008110046, 0.46749961376190186, 0.1501457393169403, 0.3712449371814728, -0.06524841487407684, 0.8817734718322754, -0.52823406457901, 0.5413344502449036, 1.2076396942138672, 0.9507015347480774, -0.38128596544265747, 0.0287232156842947, -0.9086890816688538, -0.6102697253227234, -0.5619123578071594, -0.29221442341804504, 1.0222282409667969, -0.5393794775009155, -0.8185233473777771, 0.8340932130813599, 1.1741615533828735, -0.40019702911376953, 1.558424949645996, -0.38987988233566284, 1.0163379907608032], [-0.035040777176618576, 0.5952108502388, 0.12746880948543549, -0.05776159092783928, 0.49922284483909607, -0.7544301152229309, 1.7556564807891846, -0.08282627165317535, -1.9663513898849487, 0.3582004904747009, 0.21802130341529846, 0.16649949550628662, -0.24022546410560608, 1.9252190589904785, 1.0520005226135254, 0.2456073760986328, 1.5255591869354248, -0.5383493304252625, -0.35935673117637634, 0.5145168900489807, -0.5748416185379028, -0.4079881012439728, 0.7795331478118896, -0.07210369408130646, 1.6359058618545532, -1.8264968395233154, 0.3072110712528229, -0.11801548302173615, 1.070203185081482, 1.5805702209472656, 0.29237449169158936, -0.8577573299407959, 1.5647228956222534, 1.0937422513961792], [1.573893427848816, 0.9697441458702087, -2.2305190563201904, -0.48981863260269165, 0.21462176740169525, 1.4840999841690063, -1.9141184091567993, -0.6392586827278137, -0.4665382504463196, -0.19344814121723175, 1.5504697561264038, -0.012184609659016132, -0.2838137745857239, 1.1255602836608887, 0.12946799397468567, -0.8482903838157654, -1.1963529586791992, 0.5949546694755554, -2.6612448692321777, -0.2538623809814453, -1.0681699514389038, 1.570355772972107, 0.12321532517671585, -1.4714561700820923, 1.8094103336334229, -0.6243630051612854, -0.08818767219781876, 0.873725414276123, 0.7529582977294922, 0.31644782423973083, -0.7319878935813904, -0.6767411828041077, 0.5262414813041687, -0.9004314541816711], [0.17892734706401825, -3.0940823554992676, 0.9116648435592651, -0.19273163378238678, -1.0161412954330444, -1.0525405406951904, 0.35892239212989807, 0.9522421360015869, -0.16458018124103546, 0.4592176079750061, 0.29986557364463806, 1.12577486038208, -0.9956310987472534, 0.3856440484523773, -0.34777262806892395, 0.1950906217098236, -0.718638002872467, 0.2075410932302475, -1.3404701948165894, -0.054284125566482544, 0.36921006441116333, 0.2989664673805237, 1.043425440788269, -1.1116306781768799, -1.0411568880081177, 0.5289586782455444, 1.077406406402588, 0.4793633222579956, -0.45174404978752136, -0.6773245334625244, 1.2101397514343262, -2.955033302307129, 0.3517819941043854, 0.5248890519142151], [0.6193976998329163, 0.33152657747268677, -0.3011811673641205, 3.1346137523651123, 2.726510524749756, 1.1897321939468384, 0.700579047203064, -0.7782137393951416, 0.5281949043273926, 0.702576220035553, 1.1767140626907349, -1.740996241569519, 0.20317882299423218, 0.14774660766124725, 0.23588629066944122, 0.566521406173706, -0.5937467217445374, -2.0164783000946045, 0.5132705569267273, -0.43907198309898376, 0.21440112590789795, 1.2970187664031982, 0.7539018392562866, -0.8236703872680664, -0.4878551661968231, -0.35926949977874756, 0.09415280818939209, 0.25098028779029846, 2.1231846809387207, -3.0182206630706787, -0.22650288045406342, 0.6462637782096863, -0.971717357635498, -0.5967943072319031], [-0.23754733800888062, 0.3431915044784546, 0.3176189661026001, -0.051593583077192307, -0.9875621199607849, 0.8543993830680847, -0.937459409236908, 1.6285628080368042, -0.7195920944213867, -0.9520783424377441, -0.25157228112220764, -1.1766260862350464, -0.5289705991744995, -1.0444165468215942, 0.43376660346984863, 1.043398141860962, -0.7849991917610168, -0.28107044100761414, -0.2819984555244446, -0.37952786684036255, -0.00046242887037806213, 0.8791860938072205, -1.2595714330673218, -0.11092464625835419, 0.7749629020690918, 1.1758228540420532, -0.4429885149002075, 0.7888039350509644, 0.7526717185974121, -0.4706360697746277, 0.6675481200218201, 0.27876657247543335, 0.654269278049469, 1.7838592529296875], [-0.5736434459686279, 1.0498334169387817, -0.45096778869628906, 0.26983967423439026, -0.24901443719863892, -0.09849578142166138, -0.6728174090385437, 0.20016981661319733, -0.05781738832592964, -1.0629916191101074, -1.421176791191101, -0.6311625242233276, 0.15738056600093842, -0.31211236119270325, -0.3182183504104614, 0.7061629295349121, -1.0758980512619019, -0.675434410572052, 0.05832953378558159, -2.195357322692871, 0.9968307018280029, 0.5899203419685364, 1.814016580581665, -1.2010058164596558, -0.7687914967536926, 0.5865898132324219, -0.3685358464717865, -0.2523987591266632, 1.351275086402893, -0.7038997411727905, -0.6761739253997803, -0.8059744834899902, -0.6446698904037476, 0.9873017072677612], [0.4383794963359833, -0.1078421026468277, 0.6065751910209656, 0.4961186945438385, -0.8960236310958862, -0.3813963830471039, -0.004487823229283094, 0.9767136573791504, 0.4514375627040863, -0.9175979495048523, -1.2418887615203857, 1.7043803930282593, -0.32433003187179565, -0.7400748133659363, 0.15388013422489166, -0.8773704171180725, -0.30766335129737854, -0.5336198806762695, -0.47601741552352905, 1.3590723276138306, 0.11538346856832504, -0.41015195846557617, -0.9279851317405701, -0.08887161314487457, 0.9986425042152405, -0.24512150883674622, -2.450101613998413, -1.9876365661621094, -0.19224081933498383, -0.47597241401672363, -0.5562683343887329, -1.2844771146774292, 0.6035823225975037, -1.709118366241455], [2.3018767833709717, 0.8606343269348145, -1.35233736038208, -2.0482065677642822, 1.3444005250930786, -0.5420063138008118, -0.880388617515564, 0.35714730620384216, 0.6061716675758362, 0.0444621667265892, 0.40361565351486206, 0.5140711665153503, 1.0496337413787842, -1.0406999588012695, 1.7928651571273804, 2.1663620471954346, -1.0872814655303955, -0.6007893681526184, -1.0904117822647095, 1.1219079494476318, 0.8386157751083374, -0.13198570907115936, 0.7962548732757568, 0.7190285921096802, 1.3030824661254883, -0.17084430158138275, 1.666210651397705, 0.3338639438152313, -0.7433342933654785, -0.3648265302181244, -0.7743154764175415, -0.6489192247390747, 2.1888740062713623, -0.18942657113075256], [0.8218777179718018, 0.4029194116592407, 0.16689391434192657, 0.9332000613212585, -1.3476622104644775, 0.5756961703300476, 0.7077280282974243, 0.8900744318962097, -1.5233511924743652, 1.0850287675857544, -1.080931544303894, -0.3456759452819824, 0.9256750345230103, 1.3260804414749146, 0.7599836587905884, 1.1225976943969727, -0.6812602877616882, -0.8134775757789612, 1.321306586265564, 0.7278915047645569, -0.958995521068573, 2.137660026550293, -0.9901204109191895, 2.004265546798706, -1.6713852882385254, -1.6539149284362793, -0.8839742541313171, -1.1431492567062378, 0.12982438504695892, 0.5407191514968872, 1.0132848024368286, 0.7883248329162598, 0.4059220254421234, -0.005219430197030306], [-0.3218505084514618, -0.35308486223220825, 0.8120167255401611, -1.071264386177063, 1.9725306034088135, -1.3352282047271729, 0.20751510560512543, -0.3330315351486206, -0.47414717078208923, 0.5260136127471924, -0.7550187706947327, 0.4592823088169098, 1.9171419143676758, -0.9491880536079407, 0.10437002032995224, -0.15401691198349, -0.353031724691391, -0.27594292163848877, 1.9363943338394165, 1.343736171722412, -0.24622444808483124, -0.2208968847990036, -0.42955219745635986, -0.2017323523759842, 0.7538655400276184, -0.8439998030662537, 1.948870301246643, -1.3671754598617554, 0.06403996795415878, -0.33664947748184204, -1.0622011423110962, 1.9167730808258057, 0.13544437289237976, 1.0991835594177246], [-0.21512563526630402, -1.2164404392242432, 2.557701349258423, 0.4200892448425293, 0.20850425958633423, -0.3404567539691925, 1.3343538045883179, -1.7953969240188599, 1.8644391298294067, 0.1845991611480713, -2.0519771575927734, 0.2701128125190735, 0.9834960103034973, 1.376055359840393, -0.05669187009334564, 1.4898467063903809, -0.9991217851638794, 0.054141685366630554, -1.0110300779342651, 0.13413767516613007, -0.15102335810661316, -2.214582920074463, -0.08414189517498016, 0.3537558615207672, -0.10349877178668976, -0.8361309766769409, 0.17994701862335205, -0.1700710505247116, -0.43385595083236694, 0.06789074838161469, -0.09758072346448898, -0.8990565538406372, -0.8808910846710205, 1.3296111822128296], [-1.601183533668518, -0.3545879125595093, -0.44870269298553467, 0.6777913570404053, -0.458750456571579, 0.22949349880218506, 0.04424479231238365, 0.9598440527915955, 0.019731555134058, -0.4080714285373688, -1.0333752632141113, -0.3571265637874603, 0.6432374715805054, 0.16706404089927673, -1.299992561340332, 0.39834654331207275, -1.017372727394104, 0.2573480010032654, 0.4933868646621704, -1.8072929382324219, 0.4630233943462372, 0.4584246575832367, -2.506197214126587, 0.9664510488510132, -1.100574254989624, -1.2600432634353638, -1.1748889684677124, -1.2907804250717163, 0.7147019505500793, 0.5511207580566406, 1.856405258178711, 1.147403597831726, 1.0193694829940796, 0.7227311730384827], [-1.3676084280014038, -2.0138015747070312, -0.560678243637085, 0.7104684710502625, 0.0838790014386177, 0.9974854588508606, -1.1807022094726562, 1.0730226039886475, -0.614332914352417, -2.621352434158325, -1.3185452222824097, 0.0030628382228314877, 0.15429887175559998, -0.15639932453632355, -0.2490864396095276, -0.46704167127609253, 0.5640993714332581, -1.1170769929885864, 0.1960678994655609, -0.5029159784317017, 0.007649505045264959, 0.10503095388412476, 0.7546095848083496, -0.7731079459190369, 0.03432401642203331, -1.5935570001602173, 1.2485456466674805, 0.061895616352558136, 0.11064127087593079, 1.326338529586792, 1.2476372718811035, 0.8789028525352478, 1.7215838432312012, 1.712661862373352], [0.38301628828048706, 0.5971292853355408, 0.6182491183280945, 0.06786957383155823, 0.4331184923648834, -1.3769809007644653, 1.4823564291000366, 0.912006139755249, -0.13357731699943542, 1.1648519039154053, -1.1412385702133179, 0.12300416827201843, -0.12518423795700073, -1.1123929023742676, 0.07282174378633499, 1.1900867223739624, 0.7356680631637573, -0.7821297645568848, -0.40280792117118835, -0.3875997066497803, -0.4792366623878479, -0.578929603099823, -1.5379219055175781, 0.033190011978149414, 1.5575038194656372, -0.7512875199317932, -0.02914404682815075, -2.9884872436523438, -0.27666229009628296, 2.044276475906372, 1.005786418914795, -1.2903714179992676, 0.2573607861995697, -0.44336262345314026], [-2.5881004333496094, -0.08151326328516006, 0.20629872381687164, -1.6370069980621338, 0.1671161949634552, 0.7749152779579163, -2.0788326263427734, 0.4735705554485321, -0.9636755585670471, -0.9702300429344177, 1.436290979385376, -0.28842365741729736, 0.24412420392036438, 0.5387144088745117, 1.3139827251434326, 1.5518790483474731, 0.3461579382419586, -0.4835220277309418, 1.15638267993927, -2.6086442470550537, 0.7242967486381531, 1.106943964958191, -0.6082731485366821, -0.10826550424098969, -2.0216424465179443, -0.756640613079071, -1.2602704763412476, -0.650340735912323, -1.8124420642852783, 0.7570880651473999, -0.8032220602035522, -1.0490061044692993, 1.2588053941726685, 1.7668578624725342], [0.22097156941890717, -0.757278561592102, 1.0520659685134888, -0.24536600708961487, -0.30403876304626465, 1.5585054159164429, -1.1012529134750366, -0.3368510603904724, 0.22857628762722015, -0.7637760639190674, -1.5930169820785522, 0.6179818511009216, 0.5461876392364502, -0.6694444417953491, -0.7737668752670288, 1.948852777481079, -1.8331958055496216, 0.6533987522125244, 0.2734021246433258, 0.569997251033783, -0.49293917417526245, -0.18829554319381714, 0.017904266715049744, -0.9493144750595093, -1.0796608924865723, -0.046467650681734085, -0.7672131061553955, 0.027291348204016685, 0.799446702003479, -0.818120539188385, 0.35907667875289917, 1.9700695276260376, -1.0066099166870117, 0.3059953451156616], [0.00552632100880146, -0.5840350389480591, 0.8005697727203369, 0.8907752633094788, -0.14629283547401428, -0.19323308765888214, -0.8007221817970276, 0.8315984606742859, -0.10309885442256927, 0.8602084517478943, 1.95144784450531, -0.30940914154052734, -0.2093304693698883, -0.033616647124290466, 0.44113531708717346, 0.1354629397392273, 0.718442440032959, -1.671545147895813, -0.995171844959259, -0.29886481165885925, -0.5909180045127869, 0.21867820620536804, -0.9949225187301636, -0.8065724968910217, 1.3434598445892334, -0.8384756445884705, 0.6934658885002136, -2.3805463314056396, -0.691497802734375, 0.4033164083957672, -1.0695624351501465, -0.4266529679298401, -1.2119011878967285, -0.33726394176483154], [-0.8188469409942627, -0.12043420970439911, -1.1128349304199219, 1.0827155113220215, 0.46277061104774475, -0.5562852621078491, -1.3352186679840088, -1.8097262382507324, 0.4226868748664856, 0.10465932637453079, -1.0033109188079834, -1.870857834815979, -1.3740135431289673, 1.7177224159240723, -0.26768723130226135, 1.137550950050354, -1.1537280082702637, 0.8833054304122925, 1.7953505516052246, -0.41889557242393494, -1.1508212089538574, 0.8911808729171753, 0.7799133062362671, -0.4103161096572876, 1.2030235528945923, -0.567055881023407, -0.7709242701530457, -0.06813162565231323, -0.3492172062397003, -0.7592324614524841, 1.4472084045410156, 0.05942678451538086, 2.0446462631225586, 1.5357320308685303], [0.5740538239479065, -1.9868831634521484, 0.39004868268966675, -0.3804214894771576, -1.6534711122512817, 0.22041718661785126, -0.19942033290863037, -0.0441904217004776, 0.5798241496086121, -1.4066779613494873, 0.42851606011390686, 1.8394973278045654, 0.9517368078231812, -0.05717063322663307, 0.12130887061357498, -0.11086537688970566, 0.6907007694244385, 1.3856773376464844, 0.7752544283866882, 0.5764832496643066, -1.108777642250061, -0.034324537962675095, 0.19807736575603485, 2.0946176052093506, 0.4825768768787384, 0.21462759375572205, 1.4157252311706543, -1.0691394805908203, 1.1608412265777588, 1.0945948362350464, 0.1739884316921234, 0.7378395199775696, -0.048293765634298325, -1.7738332748413086], [1.2368172407150269, 0.5656284689903259, 0.6487888693809509, 1.2942944765090942, 0.2682574391365051, -0.8285585641860962, 0.6444238424301147, -0.04251261055469513, -0.5103865265846252, -0.35531821846961975, 0.2984386682510376, -1.1734167337417603, -1.3816802501678467, -0.05664129927754402, 0.47116050124168396, 1.433421015739441, -1.1399180889129639, 1.0444000959396362, -0.35057854652404785, 1.9316890239715576, 0.5417066216468811, 1.7890335321426392, -0.03271191939711571, -0.24313151836395264, 1.7587509155273438, -0.275396466255188, 0.2306412309408188, 0.0017403578385710716, 0.6450824737548828, -0.08217279613018036, -0.6626438498497009, -0.507867693901062, 0.5455959439277649, -1.2659720182418823], [-0.4955423176288605, 0.4709795117378235, 0.6515215039253235, -0.2935779094696045, -0.3339451253414154, -0.35563385486602783, 1.1894234418869019, -0.0629076287150383, 2.3919568061828613, -1.2947837114334106, 0.05746069550514221, 0.439359188079834, -1.5022780895233154, -0.16355548799037933, 1.284966230392456, -2.064462661743164, 0.5482813119888306, -0.2874714732170105, 0.5903705358505249, 0.5578701496124268, -0.7835567593574524, -0.34198901057243347, -0.04445624724030495, -0.9973753094673157, 1.4816864728927612, -0.8584451079368591, -0.8673537373542786, -0.2138141393661499, 0.7743707895278931, -0.44706472754478455, 1.125416874885559, 0.549857497215271, -1.8421238660812378, -0.0015608820831403136], [-1.1372277736663818, -0.28572529554367065, -1.146592140197754, 0.9867147207260132, 1.7723007202148438, -1.205022931098938, -1.0455455780029297, -0.5938570499420166, 0.42418158054351807, 0.8337845802307129, -0.11531328409910202, 1.717731237411499, 0.7970772981643677, 0.7278982400894165, -0.35166171193122864, 0.07188205420970917, 1.0148835182189941, 2.3726232051849365, 1.4436485767364502, -0.6340845823287964, 1.4879868030548096, 0.7265191078186035, -0.3045346438884735, -0.2654699981212616, 1.4601566791534424, 0.3540659248828888, 2.0144574642181396, 1.8164609670639038, -0.45675376057624817, 0.9714917540550232, -1.6866209506988525, -0.4460318386554718, 0.5526298880577087, -0.7436366081237793], [1.3913546800613403, -0.9349176287651062, 0.37306004762649536, -1.2547153234481812, 0.1572064608335495, 0.010093994438648224, 0.6439868211746216, 0.6968312859535217, 0.7739938497543335, 0.5153402090072632, -0.6544213891029358, -0.3274845480918884, 0.29845136404037476, -0.7732360363006592, -0.05139236897230148, 0.39908093214035034, -0.5372447371482849, -0.4162357449531555, -0.5595189929008484, -0.9442381262779236, 0.6441706418991089, 0.4419615864753723, -0.19997446238994598, -1.119460940361023, -1.1415958404541016, -0.20722055435180664, -0.8786142468452454, 0.3160378336906433, -1.0066624879837036, -1.4836516380310059, 0.6720772385597229, -0.06511904299259186, 0.6599230766296387, 0.861678421497345], [1.1022759675979614, 1.0001171827316284, -1.2802236080169678, -0.01175728626549244, -1.0121551752090454, -0.5055971741676331, 1.3910777568817139, -0.39785394072532654, -0.6886685490608215, 0.2791486978530884, 0.29205164313316345, -0.4208177924156189, -0.21753530204296112, 2.2137062549591064, 0.012344297021627426, 0.3886776566505432, -0.31169936060905457, -0.8626010417938232, 0.6445609927177429, -0.48883357644081116, -0.6891653537750244, -0.32397693395614624, -0.15311434864997864, -0.1346931755542755, -2.371285915374756, -0.914154589176178, 0.9547610282897949, 1.5401504039764404, -0.9752386808395386, 0.35086774826049805, 1.4376075267791748, 0.7473836541175842, -1.0122885704040527, -0.33888065814971924], [0.20374153554439545, -0.6270052790641785, 0.639092206954956, -0.4989282488822937, 0.2507648169994354, -0.6429176926612854, -0.8838814496994019, -0.4499991834163666, 2.143839120864868, -1.065494418144226, -1.6128287315368652, 0.3824830651283264, -0.03052840568125248, -0.9031389951705933, -0.8185712099075317, 1.0077908039093018, -0.03217572346329689, 1.1102876663208008, -0.2950703501701355, -0.5351009964942932, 1.2036677598953247, -1.2179638147354126, 0.5412517189979553, 0.0583890862762928, -1.1511337757110596, 0.05647443234920502, -0.08548160642385483, 0.6022104024887085, 0.7678248882293701, 0.06276299804449081, 1.0174190998077393, 0.42294004559516907, -1.7826372385025024, -0.22497965395450592], [-2.0132498741149902, -0.6314597129821777, -2.4821019172668457, -1.0821954011917114, -1.1705121994018555, -1.0998235940933228, -2.194666862487793, -0.6371350884437561, -0.5384899973869324, 0.6087260246276855, 1.0802865028381348, -0.8744001388549805, -0.9090505838394165, -0.5520679950714111, 0.895114541053772, 0.5941712260246277, 0.6071454882621765, -2.0421760082244873, 1.7258085012435913, 2.57538104057312, -1.0186328887939453, 0.13928857445716858, -1.1482661962509155, -0.24948619306087494, 0.8762879967689514, -0.5031785368919373, 0.08869767934083939, -0.09760148078203201, -0.299831748008728, -0.2718503475189209, -0.3284379541873932, 1.2803279161453247, 0.27318447828292847, 1.107269525527954], [-0.0008258683374151587, 0.19998258352279663, -0.20615343749523163, -0.12814676761627197, -0.7442841529846191, 2.1127712726593018, 1.9789916276931763, -0.580656886100769, 2.2109696865081787, 0.9104776978492737, 1.4895516633987427, 1.2238138914108276, 1.292038083076477, 0.19981904327869415, 1.0206515789031982, -1.2408801317214966, -0.27278760075569153, -0.26505720615386963, -1.4828749895095825, -0.7194063067436218, 0.13079524040222168, 0.207317054271698, -0.11576702445745468, 0.19450266659259796, -0.5535841584205627, 1.619277834892273, -1.1184475421905518, -0.9799849987030029, -2.1116867065429688, -1.7253834009170532, 0.6762378215789795, -0.21002227067947388, -0.305883526802063, 1.7878810167312622], [-0.7272430062294006, 1.7540204524993896, 1.5664745569229126, 0.4455082416534424, -1.375002145767212, -1.4717411994934082, 1.725412368774414, -0.16971750557422638, 1.3606030941009521, -1.0016658306121826, 0.29208752512931824, 0.5685283541679382, -1.152893304824829, 1.3880910873413086, -0.5998701453208923, -0.9255794882774353, 0.49931415915489197, 0.12372279167175293, 2.420283317565918, 0.441110223531723, 2.2471673488616943, 0.3039829730987549, 0.1678560972213745, 0.9572935104370117, -0.05547742545604706, -0.8358983397483826, -0.39062726497650146, -0.12553785741329193, 0.5610672831535339, -1.065361499786377, 2.0235915184020996, -1.9034415483474731, -0.7102330923080444, -0.8161463737487793], [0.8952890038490295, -0.5792700052261353, -0.12099745869636536, 0.40953704714775085, -0.013527357019484043, 0.8367902636528015, -1.2356159687042236, 0.665029764175415, -0.9301642179489136, -0.6301817297935486, 0.7952548265457153, -0.04261371120810509, -0.6965539455413818, -1.3817224502563477, 0.434116005897522, 1.3916531801223755, -0.662670373916626, -1.9533653259277344, -0.8680800199508667, -0.6446902751922607, 1.4756724834442139, -1.4936572313308716, 0.7093032598495483, 1.4926185607910156, -1.0020925998687744, 0.31022441387176514, 0.5392424464225769, 0.6309337615966797, 0.5920286178588867, -1.2140558958053589, -0.14480328559875488, 0.24801233410835266, 1.5062925815582275, 1.1850093603134155], [0.5396197438240051, 1.1407510042190552, 0.2359805852174759, -0.7302895784378052, 0.21713882684707642, 0.10855412483215332, -0.850070595741272, -1.4846419095993042, -0.44725707173347473, -0.9769951701164246, 0.8587498664855957, 0.8272080421447754, 0.8500857353210449, -1.588198184967041, 0.5271221995353699, -0.8123254179954529, 0.7989038825035095, -0.03605605289340019, -0.09321042150259018, -2.3096237182617188, 0.3103753924369812, 1.5262908935546875, -0.698223352432251, -1.9944771528244019, 2.3728275299072266, 0.8255826830863953, 0.45190829038619995, -2.103339672088623, -0.5703595876693726, 0.567857027053833, -0.05403147637844086, -0.4580457806587219, 0.00401817262172699, -0.07607042044401169], [0.7231261730194092, 0.7850359678268433, -0.3481859266757965, 0.16667485237121582, 0.608181893825531, 0.450003445148468, 1.2128477096557617, 1.7213913202285767, 0.27243950963020325, -1.443835735321045, 0.7054301500320435, 0.6331973671913147, -1.5586657524108887, 1.9313032627105713, 0.9965271949768066, 0.8939647674560547, 0.1457889825105667, 1.7893484830856323, -0.8085512518882751, -0.6271746158599854, 0.8955715894699097, 1.67962646484375, -1.1860435009002686, -1.540776014328003, -0.2734198272228241, -1.0100078582763672, -0.4167496860027313, 1.7418773174285889, 0.3896942436695099, 0.6490477919578552, 1.2064088582992554, 0.09318696707487106, 0.8558798432350159, -0.012852076441049576], [-1.0103418827056885, -0.5366378426551819, 1.4112646579742432, 0.18493376672267914, 1.126002311706543, -1.4367204904556274, -1.6644741296768188, -0.325054794549942, -0.2780860364437103, 0.13704314827919006, 0.07114111632108688, 0.4391705393791199, 0.46104976534843445, 0.8816680908203125, 1.3842313289642334, 0.142572283744812, -0.8751854300498962, 3.1274120807647705, 0.3794267177581787, 0.3886922597885132, 0.011081800796091557, 0.5463656187057495, -0.8388122916221619, -1.734428882598877, -0.2771931290626526, 1.9901268482208252, -1.9964542388916016, 1.9083983898162842, 0.8526675701141357, -1.2269375324249268, -0.11498840153217316, 0.46851998567581177, 0.9333178400993347, 1.477668046951294], [0.44814807176589966, -0.37623000144958496, -0.08799713104963303, 0.8251703381538391, 1.2394553422927856, -1.1003087759017944, 0.7408607602119446, 0.3185395896434784, 1.3043636083602905, 0.6780306100845337, 1.200079083442688, -0.6576220393180847, -0.07582543790340424, 1.665641188621521, 1.0583699941635132, -0.950085461139679, -0.005376553162932396, -0.27472880482673645, -0.022583039477467537, 0.2818860411643982, 0.24399495124816895, 1.0264769792556763, 1.1289161443710327, 0.031444624066352844, -0.8274876475334167, -0.22947700321674347, -0.7916391491889954, -0.8042657971382141, 0.06360791623592377, 1.1778556108474731, 0.7831897735595703, -0.3711766004562378, 0.687148928642273, -0.8239974975585938], [0.5812304615974426, 0.46779873967170715, 0.050808995962142944, -0.581944465637207, -0.12546424567699432, -1.0612621307373047, 1.5835984945297241, 1.8360788822174072, -1.1196705102920532, 2.3356430530548096, -1.2799859046936035, 0.19209077954292297, -0.3944242596626282, -2.3133938312530518, 0.8129440546035767, 0.8102415800094604, -1.0482149124145508, -1.7671175003051758, 0.5220046639442444, 0.9100627899169922, 0.6907644271850586, 0.9776194095611572, 1.0604782104492188, 0.7102636098861694, 1.1638151407241821, 0.3511674106121063, 0.2806409001350403, 0.29171547293663025, 0.6530759334564209, 0.8808895945549011, -0.717746376991272, -1.8241451978683472, 1.6287041902542114, -0.0863255113363266], [-0.8253540396690369, -0.683837354183197, 1.3898541927337646, -0.5673708915710449, -0.5834824442863464, 0.535514235496521, 1.357664942741394, -1.2142689228057861, 0.5385726690292358, 0.5664095878601074, -0.4526583254337311, -0.45389363169670105, 1.2701719999313354, -0.06010150536894798, -1.630573034286499, 0.8345932960510254, -0.886215329170227, 0.6264402270317078, 0.6879726052284241, 0.556201159954071, -3.0504448413848877, -0.8170006275177002, 0.5343596935272217, 0.20649908483028412, -0.6315931081771851, 0.4475424289703369, 1.5122531652450562, 1.0297954082489014, 0.9920854568481445, -1.1388397216796875, -0.2043914496898651, -1.3564659357070923, 0.6331842541694641, -0.7905383706092834]], "dtype": "float32"}, {"__nd__": [[0.15411126613616943, 0.3817991316318512, -0.7400479912757874, 1.4317072629928589], [-1.660945177078247, -1.2456780672073364, 0.8460640907287598, 1.61781907081604], [-0.35557863116264343, -0.7699670791625977, -0.8983440399169922, -1.1850382089614868], [0.062274158000946045, -1.4135830402374268, 0.3690609931945801, -1.0269221067428589]], "dtype": "float32"}], "e": {"__nd__": [[-3.9040770530700684, 1.430937647819519, -3.279630661010742, 0.45223772525787354, 6.364703178405762, -3.253063917160034, -1.594042181968689, 4.692498207092285, -3.292124032974243, -0.9782187938690186, 2.1319549083709717, 4.023862838745117, 0.002545967698097229, 2.983992099761963, -5.769248962402344, -2.771270275115967, -0.8721970915794373, 4.259883403778076, -1.2732948064804077, 0.9758870601654053, -4.772582530975342, 0.09985184669494629, 2.2056777477264404, 4.680901050567627, -3.577984571456909, -1.7880003452301025, -0.17830175161361694, -5.509704113006592, 0.6651239991188049, -0.3241852819919586, 0.21188655495643616], [3.991452932357788, 1.1824004650115967, -0.5030524730682373, -6.625342845916748, -3.3314943313598633, -3.0454254150390625, 2.501206636428833, 1.943498969078064, -1.2482280731201172, 0.1897445023059845, -3.722895622253418, -0.5143406391143799, 7.437751770019531, 3.3780040740966797, 2.3382651805877686, 1.7817833423614502, -0.3830864131450653, -2.2260658740997314, 4.368646621704102, 2.583773374557495, -1.4713842868804932, -1.7870705127716064, 4.891489505767822, 5.637528896331787, 5.687209129333496, 2.1223559379577637, -2.387990951538086, -2.2194390296936035, -3.761870861053467, -1.9383573532104492, -3.10251784324646], [-3.1088991165161133, -4.372766494750977, 8.901244163513184, -2.9217019081115723, -1.9547407627105713, -5.781493186950684, 3.2810182571411133, 4.6514787673950195, -6.30521297454834, -2.2255802154541016, -4.328973770141602, -2.6766583919525146, -4.995968341827393, 0.3686736822128296, 0.6078301072120667, -6.446193218231201, -4.838738441467285, 4.842369556427002, 7.27110481262207, 0.22859907150268555, 0.07590847462415695, -7.348369598388672, 0.6691325902938843, 2.4010937213897705, 10.403053283691406, -1.95914888381958, -0.01942920684814453, -1.3728580474853516, -7.6062774658203125, -2.548802137374878, -1.6682219505310059], [-2.437697172164917, 4.2224202156066895, -0.4503791332244873, -1.2155256271362305, -10.041250228881836, 7.430508613586426, 9.05606460571289, -2.6812398433685303, 0.7279003858566284, -1.4183911085128784, -5.263854026794434, -3.467212677001953, -3.0831027030944824, -2.8487095832824707, -1.755704641342163, 4.624973297119141, -2.024493455886841, -0.021690845489501953, -3.204495906829834, 1.2991437911987305, -3.4768013954162598, -6.247735023498535, 8.967391967773438, -2.517232894897461, -2.7667081356048584, -0.3575827181339264, 0.7150241136550903, -3.1909961700439453, -0.8767356872558594, -2.478752374649048, -1.1394339799880981], [6.002436637878418, 2.409087896347046, -1.3502298593521118, 3.475308656692505, 1.282564640045166, 1.2492924928665161, -5.658536434173584, 3.2848961353302, 0.6514074802398682, -0.2379969358444214, -1.853878140449524, 5.864381790161133, -2.9009344577789307, 2.6657731533050537, -0.1698806881904602, 3.5706872940063477, -0.15810361504554749, 6.559518814086914, -2.4488441944122314, -1.6020855903625488, 3.2412734031677246, -0.35639524459838867, -0.49211472272872925, -4.406497478485107, -2.826425313949585, 3.7975761890411377, 1.9929490089416504, -1.7079675197601318, 3.5761289596557617, -5.6363091468811035, 11.172164916992188], [-8.489021301269531, 2.5773844718933105, 2.764601707458496, -0.6094064116477966, -7.221227645874023, -5.8845295906066895, 4.834828853607178, 0.5472080707550049, 1.5820691585540771, -5.1047139167785645, 3.9736218452453613, 1.3995206356048584, -2.4749910831451416, -0.6816461682319641, 0.586736798286438, 1.0907301902770996, 4.433880805969238, -1.7037461996078491, 5.587869644165039, 2.162034034729004, -4.060288906097412, 2.085639476776123, 0.6095740795135498, 1.7781742811203003, -3.0487422943115234, 1.4034905433654785, 5.6707234382629395, -8.181519508361816, 2.7133169174194336, 6.81068229675293, 1.4063372611999512], [1.3294790983200073, -3.2013981342315674, -9.716317176818848, -4.998364448547363, 1.0637774467468262, 2.9726450443267822, -4.254751682281494, 0.4533945918083191, 3.159467935562134, -1.0570300817489624, 3.0263123512268066, -1.7298831939697266, -0.2668699622154236, -1.5935208797454834, 2.4495177268981934, -3.4403395652770996, 3.8189890384674072, 0.3143690824508667, 3.663698434829712, -0.33396995067596436, -6.034061431884766, 0.8221832513809204, -2.7622015476226807, 6.7999162673950195, 1.5948216915130615, -4.713077068328857, -2.5877130031585693, -0.32705652713775635, -0.6742820739746094, -1.0133084058761597, -0.011744856834411621], [-1.0969548225402832, 7.354888916015625, -0.45974060893058777, -3.442134141921997, -7.862223148345947, -2.73223876953125, 4.186956882476807, 5.5958571434021, 2.198268413543701, -2.086050033569336, 6.2807536125183105, 3.3549976348876953, 0.09957194328308105, -0.44091999530792236, -4.653120040893555, -1.318334698677063, 8.467166900634766, 1.4743996858596802, 4.121743202209473, 1.8316481113433838, -3.141951084136963, -7.145839214324951, -0.5876659154891968, 2.9247026443481445, -0.7975871562957764, 1.0171785354614258, -2.8548474311828613, -6.566052436828613, -3.5453877449035645, 7.222289085388184, -5.975358009338379], [4.460198402404785, -0.8214209079742432, 0.6098759174346924, 4.311669826507568, 1.6696045398712158, 2.066561698913574, -1.0701628923416138, 1.197187066078186, -0.7687065601348877, 6.693492412567139, -1.3023946285247803, 3.3631839752197266, 6.576005935668945, 0.37582069635391235, -2.3134071826934814, 2.4256863594055176, 1.5471340417861938, 3.0222792625427246, 2.5608835220336914, -2.4726638793945312, -4.227778434753418, 2.2788376808166504, 5.399904251098633, 1.5216739177703857, 0.22426295280456543, 4.931154727935791, -3.132974863052368, 2.2191858291625977, 4.018291473388672, 0.745125412940979, 6.089136123657227], [-1.1891124248504639, -2.5419063568115234, 6.549297332763672, -4.131764888763428, 3.107361078262329, -1.9374363422393799, -1.7039602994918823, -2.769961357116699, -3.538313388824463, 1.8069326877593994, 2.235292911529541, -2.466728448867798, 3.2532668113708496, -1.5054024457931519, -2.915789842605591, 3.5178332328796387, -2.872570514678955, 1.1891487836837769, 3.8027472496032715, 2.239922523498535, -0.6818158626556396, -7.103024482727051, -0.8636130094528198, 0.4419177770614624, 6.836437225341797, 4.470932960510254, 2.803570032119751, 2.433095932006836, -0.09541082382202148, 0.8200633525848389, 9.999383926391602], [2.5003347396850586, 0.3266535997390747, -3.828674554824829, 1.6904995441436768, 3.004596710205078, 2.4524941444396973, -7.2679033279418945, -3.6020641326904297, -1.0609272718429565, 2.4595067501068115, -3.403251886367798, -6.867639064788818, -5.1160125732421875, -3.9809536933898926, -1.8089327812194824, 2.1749215126037598, 0.7086695432662964, 4.172399997711182, -6.759415626525879, -0.7095757722854614, -10.561811447143555, 5.181784629821777, -0.2879984378814697, -7.0031538009643555, -5.767803192138672, 3.4877431392669678, 4.7559003829956055, 2.871284246444702, -1.5325374603271484, -2.1295523643493652, -0.1147102415561676], [-9.044791221618652, -3.8417458534240723, 8.872709274291992, -4.356389045715332, 0.16736367344856262, 3.3863067626953125, -0.7591836452484131, 1.3478630781173706, 2.7529563903808594, -1.8239719867706299, -3.233048677444458, -5.055116653442383, 1.166684627532959, -1.4362201690673828, -7.597177028656006, -7.366049766540527, 4.015553951263428, 1.3063290119171143, -3.662407398223877, -1.9792343378067017, -1.4158776998519897, 2.9490878582000732, 1.0145916938781738, -3.5087218284606934, 6.405917167663574, -3.7253801822662354, -3.217717170715332, -2.6029553413391113, -5.842038154602051, 5.978847026824951, -5.010080337524414], [1.6806645393371582, -4.836170196533203, -4.071425437927246, -0.8496860265731812, 8.088500022888184, -6.673962116241455, 0.902822732925415, 0.7536081075668335, -1.0294206142425537, 1.7443840503692627, 0.903888463973999, 3.8717219829559326, -2.4878945350646973, -3.38179612159729, -5.313499450683594, 0.07876807451248169, 2.8130688667297363, -2.0399909019470215, -0.27795863151550293, 1.83332097530365, 5.463196277618408, -1.7932151556015015, -4.807525634765625, -3.3411550521850586, 3.2872776985168457, 2.417757511138916, 5.011776924133301, 5.488796234130859, 0.3170703053474426, 2.1956546306610107, -7.536281585693359], [-0.8211479187011719, -1.4623050689697266, -0.709765613079071, -2.110673666000366, -1.4189567565917969, -2.787537097930908, 1.7362496852874756, -0.0164949893951416, 2.0026938915252686, 5.6418046951293945, 0.197062686085701, -4.935502529144287, -4.677765369415283, 1.8732572793960571, -2.4187583923339844, 8.441744804382324, 7.064834117889404, -3.7540578842163086, 4.0977983474731445, -1.926945686340332, 3.1435935497283936, 3.74139404296875, -0.07432234287261963, 4.313180923461914, 0.6897807121276855, -0.44178205728530884, -0.0934373140335083, -1.2327651977539062, 3.1478748321533203, 2.880450963973999, -0.5109621286392212], [4.664373874664307, 4.649142742156982, -11.055953979492188, 4.14638614654541, -6.667313575744629, 1.598381757736206, 3.4910378456115723, -1.6500520706176758, 1.5544686317443848, 8.309475898742676, 3.4090280532836914, 2.055690288543701, -0.39320072531700134, -1.688683271408081, 0.06858289241790771, -1.6160590648651123, 5.349822044372559, 0.45177534222602844, -0.645979642868042, -0.316972017288208, 4.930363655090332, 5.767498970031738, 1.7672158479690552, 3.8576600551605225, 4.0834479331970215, 1.7466381788253784, -2.3097622394561768, -4.465978145599365, -3.505350112915039, -10.847705841064453, -0.8744869232177734], [3.065964937210083, 0.9094939231872559, -0.2916916608810425, -0.43446314334869385, -1.0142326354980469, 3.473775625228882, -2.698430061340332, -0.4188321828842163, 2.5018203258514404, 6.778172969818115, 5.369320869445801, -2.837214708328247, 2.332207202911377, -2.799584150314331, 3.2577247619628906, -0.3557114005088806, 2.14990496635437, -0.956439733505249, 0.5283862948417664, 1.4430725574493408, -1.855499267578125, 1.2004649639129639, 2.3788607120513916, -4.556161403656006, 3.8404717445373535, 1.6592904329299927, 5.295165061950684, 1.4908058643341064, -4.136178970336914, -7.5274882316589355, -2.5433061122894287], [6.854907989501953, 1.0606293678283691, 5.152037143707275, -1.322599172592163, -3.1312053203582764, 1.9366658926010132, -7.009649753570557, -6.7467851638793945, 5.443618297576904, 5.030901908874512, 3.307103157043457, -3.0748143196105957, -1.7622358798980713, -4.342493534088135, -1.425205945968628, -1.934268832206726, 1.7681736946105957, 1.5595452785491943, 4.625245094299316, 0.8025997877120972, 3.0543763637542725, -2.410778522491455, -3.687654972076416, 4.256534576416016, 6.194009780883789, 9.316341400146484, -0.7933866381645203, 7.097012519836426, -0.021313786506652832, 0.031319618225097656, 3.077885627746582], [4.167469024658203, -2.1038901805877686, -2.6034560203552246, 2.635180950164795, 4.897019863128662, 2.3617327213287354, -1.2310163974761963, -1.473123550415039, -2.098489284515381, -0.219967782497406, -1.9334640502929688, -3.365745782852173, -1.7673012018203735, 5.634311676025391, -10.030529975891113, -2.036893129348755, -1.640694260597229, 2.120910882949829, -1.349611520767212, -0.06833500415086746, -0.5725349187850952, 9.639805793762207, 4.134310245513916, 6.188836574554443, -5.057499408721924, 2.8940024375915527, 11.606108665466309, 9.621020317077637, -1.1232571601867676, -1.8264272212982178, -4.436445236206055], [1.9098246097564697, -0.9874221086502075, -1.9029850959777832, 4.359948635101318, -3.0746750831604004, 1.489797592163086, 2.0250494480133057, 0.5389328002929688, 6.02371072769165, -2.8616466522216797, -1.6676877737045288, 4.344440937042236, 2.5168540477752686, -0.68763667345047, -4.338788986206055, -1.8751853704452515, -2.0817389488220215, -0.9178428649902344, 3.0959348678588867, 3.6108388900756836, -0.8366100788116455, -0.9860421419143677, -0.05735063552856445, 0.2819893956184387, 2.924128770828247, -0.532767653465271, 7.822824001312256, 3.8005475997924805, -5.0445990562438965, 4.9761223793029785, 4.7341132164001465], [-4.587285995483398, 1.2190567255020142, -0.2529505491256714, -1.9174907207489014, 1.7404625415802002, -2.7252390384674072, 1.632209062576294, -3.4121150970458984, -2.3538167476654053, 6.484197616577148, 2.0403237342834473, 0.5195536613464355, -0.319491982460022, 1.4798831939697266, -3.673107624053955, 4.690155506134033, 0.7145649194717407, 2.4671242237091064, -1.4350519180297852, 0.3939340114593506, 1.6311473846435547, -7.748344421386719, 2.8583502769470215, -2.351680278778076, 3.675208806991577, 3.7480764389038086, 5.004525661468506, -2.647829055786133, 5.210935592651367, 1.6671710014343262, -0.952500581741333], [4.9313740730285645, 0.78214430809021, -0.8199281692504883, -1.6996649503707886, 5.197577953338623, 4.286062717437744, 3.1105475425720215, 0.5833269357681274, 5.23864221572876, -1.808929204940796, -5.627140522003174, 0.27210667729377747, 2.5304553508758545, -4.666821479797363, -1.8757002353668213, -8.341094970703125, -3.4638397693634033, 2.270087242126465, 0.7553268074989319, 0.5699256062507629, -5.547622203826904, -0.47727900743484497, -0.5508456826210022, -2.827651023864746, -1.8995918035507202, -4.083982467651367, 2.4541306495666504, 2.6331288814544678, -1.4136106967926025, -7.723857402801514, -0.7179244756698608], [2.3948183059692383, 4.6256256103515625, 0.8253728747367859, -5.747030258178711, -0.32714658975601196, -0.38497722148895264, 8.028736114501953, 3.7971596717834473, -6.184886455535889, -3.9911794662475586, 4.772373199462891, 6.264061450958252, 0.002594590187072754, -2.093989610671997, -9.035124778747559, 3.0972366333007812, -4.9141740798950195, -6.758533477783203, -5.581659317016602, 2.780374526977539, -4.125760555267334, -3.010465145111084, -4.437318325042725, -6.368704795837402, -5.848327159881592, -0.39975234866142273, -1.557083010673523, -2.178283452987671, 1.22153902053833, 0.10597056150436401, 5.516479015350342], [0.2909848690032959, -3.6031494140625, -1.9021872282028198, -0.18603551387786865, 1.4871318340301514, -1.6097066402435303, 0.5999736785888672, -6.325547695159912, 5.423583984375, 8.080144882202148, 1.4185281991958618, -4.815676212310791, 1.8294281959533691, -4.7946929931640625, 7.285311222076416, 1.6311497688293457, -5.341207981109619, -7.409953594207764, -5.414278507232666, -0.9423272013664246, 4.628791332244873, 0.4994661808013916, -2.1405937671661377, -4.772572040557861, -0.5762634873390198, -2.525019645690918, 1.1876239776611328, 1.9604473114013672, -1.7184338569641113, 2.655284881591797, -2.9360766410827637], [-3.1466434001922607, -2.3241450786590576, -1.2081234455108643, -5.824313640594482, 2.4130020141601562, -2.2868199348449707, -5.428966045379639, 1.8229172229766846, -1.8713603019714355, -1.2295191287994385, -2.016444206237793, 4.481964111328125, 5.7287702560424805, -0.21542608737945557, -1.5985217094421387, -2.47977352142334, 1.0361751317977905, -3.8717856407165527, 3.898684501647949, -3.2451679706573486, 0.10868275165557861, -2.9057233333587646, 3.1816415786743164, -2.2775375843048096, -5.167064189910889, 2.803391456604004, -2.7556726932525635, -0.8334375619888306, -3.986830949783325, 3.7288665771484375, 0.27764561772346497], [5.411651134490967, -5.833698272705078, -1.0366013050079346, 4.012782096862793, 2.54769229888916, 2.1616411209106445, -3.744692802429199, -4.305114269256592, -7.0260701179504395, -3.1958389282226562, -3.172823905944824, 0.953630805015564, 1.984851598739624, -5.242133617401123, -0.5968749523162842, -1.4752132892608643, 2.6824893951416016, -5.074370384216309, -1.1867327690124512, -2.9910197257995605, 0.5097561478614807, 4.691500186920166, -0.669812023639679, -2.5159430503845215, -7.65474796295166, 4.158053874969482, -1.1251943111419678, 1.2283724546432495, 4.751378536224365, -3.3281948566436768, -5.807053565979004], [1.643363356590271, 9.558541297912598, 0.7143667936325073, -5.396478652954102, -4.339851379394531, 3.877990961074829, -1.2438154220581055, 3.141526937484741, 1.8054492473602295, -1.2181761264801025, 0.35048460960388184, -0.6163623929023743, -10.087376594543457, 4.886549949645996, 4.693177700042725, 4.7017822265625, -0.03611844778060913, -4.995724201202393, 1.8536298274993896, -0.04099580645561218, -4.276224136352539, 9.03707504272461, 2.3801429271698, 6.516301155090332, 2.908703327178955, 0.3264533281326294, -5.958133220672607, -0.8231716156005859, -2.5170187950134277, -3.087855100631714, 1.57053542137146], [0.765490710735321, 2.0623016357421875, 1.7264103889465332, 3.0362462997436523, 2.1664364337921143, 0.6086442470550537, 2.608889102935791, -5.176617622375488, 0.8352534770965576, 2.817182779312134, -0.34588873386383057, -2.150203227996826, -0.32752037048339844, 2.923949956893921, -0.5496706962585449, 0.39250266551971436, -1.8597103357315063, 5.266988277435303, 6.868182182312012, -1.2501580715179443, -0.5206410884857178, 2.5305490493774414, 2.5435972213745117, 8.126313209533691, 1.454958438873291, -2.1769418716430664, -1.1561729907989502, -6.352149963378906, 2.693281650543213, 3.1156132221221924, -1.72369384765625], [-5.003829002380371, 3.5920262336730957, 2.7094337940216064, 8.92867374420166, 6.452665328979492, 0.6206455230712891, -1.477989673614502, 2.5264992713928223, 2.1502013206481934, -0.601294755935669, 4.794060707092285, 5.201451778411865, -0.7055245637893677, -6.690659523010254, -2.4524784088134766, -4.909510612487793, 1.1397721767425537, -1.2389436960220337, -5.032621383666992, 3.182755947113037, -0.45949000120162964, -2.067737579345703, -1.122102975845337, 1.6209341287612915, 9.8522367477417, -3.278167486190796, -6.041158199310303, 1.5912407636642456, -0.5109459757804871, -0.7917985916137695, -5.585439205169678], [6.145557880401611, 3.969669818878174, 0.3500988483428955, 5.186384201049805, -0.7562084197998047, 3.1588547229766846, 4.948843002319336, -11.08758544921875, -5.865347385406494, 2.0979833602905273, 4.32456636428833, -6.732667922973633, 4.161635875701904, -0.4220174252986908, 4.107240676879883, 1.174260139465332, -6.695791721343994, -0.5230253338813782, -0.9231624603271484, -0.013471588492393494, 0.815696120262146, -3.0307581424713135, -3.2231101989746094, 4.665529251098633, 1.9095335006713867, 3.4943342208862305, 5.045467853546143, 3.8977770805358887, 2.047807455062866, -7.248339653015137, -7.354562759399414], [-3.3363404273986816, 2.7631402015686035, 0.31158459186553955, -5.964503288269043, 0.5836554169654846, -2.247819423675537, 0.10442650318145752, -3.4621784687042236, -1.9507203102111816, -8.856660842895508, -9.246156692504883, 3.08388352394104, 4.250740051269531, 0.7044331431388855, -0.6651027798652649, -2.3366518020629883, 9.815497398376465, 0.11971926689147949, -8.832998275756836, -5.798388957977295, -1.9967317581176758, 1.4218933582305908, 1.148759126663208, -1.0955982208251953, 2.1371147632598877, 4.333121299743652, 5.567319393157959, 1.021233081817627, 8.588163375854492, -2.337235689163208, 3.75856614112854], [-3.5110888481140137, -3.6084439754486084, 2.4148199558258057, 4.987373352050781, -2.5984392166137695, -3.4612677097320557, 0.23718872666358948, 2.056285858154297, -2.9467475414276123, 0.13275957107543945, -2.746385097503662, -0.43793365359306335, -2.0837254524230957, -1.7057651281356812, -3.6654341220855713, 2.242426872253418, 0.18052363395690918, -6.744380950927734, 5.364233016967773, -6.483874320983887, -1.2560373544692993, -0.45619916915893555, -2.050328016281128, 2.708958625793457, -3.678600549697876, -7.014657020568848, -0.08950924873352051, 0.5685802698135376, 10.318611145019531, 0.673253059387207, 4.924063682556152], [0.12352925539016724, -8.538662910461426, -3.442261219024658, 4.622847557067871, 4.451815605163574, 9.3076171875, 1.0205103158950806, -0.17366743087768555, 2.4656412601470947, 0.20191365480422974, 2.945648670196533, 1.6917588710784912, -1.476534366607666, -2.857442855834961, 5.475072383880615, 4.414742469787598, 7.423678398132324, 0.0527595579624176, 0.5760698318481445, -3.0912275314331055, -4.085271835327148, -3.2017180919647217, 1.8487842082977295, -10.15576171875, 0.7502286434173584, -2.370908498764038, 0.5393708348274231, 4.692192554473877, -2.5848097801208496, -5.326983451843262, -2.2743594646453857], [-1.9502129554748535, -0.20299121737480164, 0.014849305152893066, 0.22231078147888184, -2.839245319366455, 2.8614206314086914, -0.5921046733856201, 1.6574013233184814, 4.186520099639893, -4.538173198699951, -4.2234416007995605, 0.018668174743652344, 3.000236749649048, 2.077967643737793, -10.035945892333984, 0.03536611795425415, 1.1108393669128418, 11.89653205871582, -0.4436049461364746, 0.6290808916091919, 4.66445779800415, 1.0300664901733398, -2.433288097381592, -1.9595425128936768, 2.611901044845581, 3.615635871887207, -6.906135082244873, 0.13283753395080566, -3.922046184539795, 3.5807509422302246, 2.79626727104187], [-2.4261298179626465, -6.911098480224609, 2.856128215789795, -5.441494941711426, -3.3046860694885254, -3.801853895187378, 0.9739662408828735, 4.927653789520264, 1.9862045049667358, 2.7140274047851562, -8.641861915588379, -5.213671684265137, -3.0060555934906006, -2.81967830657959, -7.898061752319336, 1.2681339979171753, -10.920419692993164, -0.03968380019068718, -0.9540975093841553, 6.697871685028076, -2.9854557514190674, 0.875771164894104, 12.156452178955078, 7.404523849487305, -13.863741874694824, -2.573904037475586, -3.106268882751465, 1.2581217288970947, -1.0254313945770264, -2.4833054542541504, -1.9019348621368408], [-3.9400384426116943, -1.8916242122650146, 1.8278098106384277, -0.5557488203048706, 4.983949184417725, -0.13424253463745117, -7.719249248504639, -4.409748077392578, 2.724169969558716, -2.5477335453033447, -5.1660075187683105, 3.168699264526367, -2.8042397499084473, -3.963228940963745, -2.4551289081573486, -4.153278827667236, -10.106281280517578, -0.832332193851471, 4.203036785125732, -3.6896133422851562, -7.451341152191162, 3.451218843460083, 0.06128622591495514, 5.313734531402588, 0.4176296889781952, 4.187981605529785, 0.2337961196899414, -2.5234270095825195, -1.5724247694015503, -3.58420467376709, -2.9477643966674805], [3.970094680786133, -0.29026126861572266, -2.2862112522125244, -6.970902442932129, -0.6274151802062988, 0.8950899243354797, -6.061224937438965, 4.289825916290283, -4.388554573059082, 0.0941581130027771, 5.473003387451172, -2.3958187103271484, 2.2104945182800293, -4.723782539367676, 5.809928894042969, 2.7843449115753174, 0.5042790770530701, -5.255659580230713, -2.490755558013916, -8.446030616760254, -8.369060516357422, -3.8767452239990234, 4.244760513305664, 1.4996229410171509, 3.2694284915924072, 1.5300488471984863, -0.6937901973724365, -5.383931636810303, -0.46811485290527344, 3.025463819503784, 5.947422981262207], [1.9325807094573975, 4.217781066894531, -2.2916760444641113, -4.535094738006592, -0.9364830255508423, -1.7816531658172607, -1.7261037826538086, -0.3297487199306488, -3.420149087905884, -3.263636350631714, 7.103141784667969, 7.060039520263672, -4.066901206970215, -1.361355185508728, 4.240725994110107, 1.5496671199798584, 1.050915241241455, 2.467360496520996, -1.1612203121185303, 0.8875879645347595, -4.172670841217041, -7.035477638244629, -4.059586524963379, -7.479400634765625, 2.1627838611602783, -3.9828524589538574, 0.5285542011260986, 2.7871909141540527, 3.7932631969451904, -2.3625857830047607, 3.245035171508789]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(conv2d_valid, json.loads(_TESTS), 'cuda', *[1e-05, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")